In [1]:
#@title 🧬 Peptide Property-Based Benchmarking { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 28px 30px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 10px 25px rgba(255, 182, 193, 0.4);
        margin-bottom: 20px;
    }

    .title {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 26px;
        font-weight: bold;
    }

    .toggle-btn {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 9px 16px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        margin: 8px 6px 0 0;
        transition: all 0.2s;
    }

    .toggle-btn:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header" id="headerBox">
    <div class="title">🧬 Peptide Property-Based Benchmarking</div>
    <div style="text-align:center; color:#4b5563; font-size:15px; margin-bottom:12px;">
        RL vs Pseudo-RL vs Hybrid and Advanced &nbsp;|&nbsp; vs Selected Database Reference Peptides
    </div>
    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:16px 0;">
    <div style="font-size:14.8px; line-height:1.75; color:#374151;">
        This notebook benchmarks generated peptides (RL / Pseudo-RL / Hybrid and Advanced / ...) against
        <b>selected database reference peptides</b>, on two complementary levels:

        <ul>
            <li><b>Literature-defined ranges</b> — fixed, absolute thresholds from published studies (e.g. AMP length 10–50 aa, net charge +2 to +9; no universal pI compliance interval).</li>
            <li><b>Statistical similarity to the selected task-specific database reference subsets</b> — Mann-Whitney U, Kolmogorov-Smirnov, Wasserstein distance, and empirical P5–P95 compliance.</li>
        </ul>
        Both are reported at the <b>Model</b> level and, separately, at the <b>Generation-Strategy</b> level
        (RL / Pseudo-RL / Hybrid and Advanced), since Strategy is a generation-method dimension independent of the biological Category (AMP/MHC/Cyclic).

        <br>
        <b>🟦 Blue/colored cards</b> = explanation.
        <b>⚙️ Gray code cells</b> = the code that runs it.
        Run cells top to bottom.
    </div>
</div>
<button class="toggle-btn" onclick="toggleHeader()">Hide Header</button>
<script>
function toggleHeader() {
    const box = document.getElementById("headerBox");
    const btns = document.querySelectorAll(".toggle-btn");
    const headerBtn = btns[0];

    if (box.style.display === "none") {
        box.style.display = "block";
        headerBtn.textContent = "Hide Header";
    } else {
        box.style.display = "none";
        headerBtn.textContent = "Show Header";
    }
}
</script>
"""))


In [2]:
#@title ⚙️ 1. Setup — Install & Import { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-setup {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-setup {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-setup {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-setup:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-setup" id="headerBox_setup">
    <div class="title-setup">⚙️ 1. Setup — Install &amp; Import</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Installs BioPython / pandas / numpy / scipy / matplotlib / seaborn, and imports everything used below.
    </div>
</div>
<button class="toggle-btn-setup" onclick="toggleHeader_setup()">Hide card</button>
<script>
function toggleHeader_setup() {
    const box = document.getElementById("headerBox_setup");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


# Install required packages
!pip install "biopython==1.86" pandas numpy scipy matplotlib seaborn -q

import Bio
assert Bio.__version__ == "1.86", f"Expected Biopython 1.86, got {Bio.__version__}"
print("Biopython version verified:", Bio.__version__)
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import pandas as pd
import numpy as np
from scipy import stats
import re
import io
import base64
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.4 MB/s eta 0:00:00
Biopython version verified: 1.86


In [3]:
#@title 📤 2. Upload FASTA Files { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-upload {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-upload {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-upload {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-upload:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-upload" id="headerBox_upload">
    <div class="title-upload">📤 2. Upload FASTA Files</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Upload the generated-peptide FASTA file first, then the three selected database reference subsets (one per category).
        Expected header format for the <b>main</b> file: <code>&gt;ModelName_ID (Type)</code> —
        e.g. <code>&gt;HMAMP_001 (Antimicrobial)</code>. Natural files don't need model/type tags in the header;
        their category is fixed by which upload slot they go into.
    </div>
</div>
<button class="toggle-btn-upload" onclick="toggleHeader_upload()">Hide card</button>
<script>
function toggleHeader_upload() {
    const box = document.getElementById("headerBox_upload");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


print("Step 1/4 - Upload the FASTA file with your RL/model-generated peptides:")
uploaded_main = files.upload()
main_fasta_file = list(uploaded_main.keys())[0]

print("\nStep 2/4 - Upload the FASTA file with selected APD6 AMP reference peptides:")
uploaded_amp = files.upload()
amp_fasta_file = list(uploaded_amp.keys())[0]

print("\nStep 3/4 - Upload the FASTA file with selected IEDB MHC ligand reference peptides:")
uploaded_mhc = files.upload()
mhc_fasta_file = list(uploaded_mhc.keys())[0]

print("\nStep 4/4 - Upload the FASTA file with mapped CycPeptMPDB cyclic reference peptides:")
uploaded_cyclic = files.upload()
cyclic_fasta_file = list(uploaded_cyclic.keys())[0]

NATURAL_FILES = {
    "AMP": amp_fasta_file,
    "MHC": mhc_fasta_file,
    "Cyclic": cyclic_fasta_file,
}

# Reference input quality checks, BEFORE any downstream calculation.
# The cycle reference comprises mapped parent-residue strings, not molecular structures.
from collections import Counter as _ReferenceCounter
REFERENCE_LABEL = "Selected database reference"
for _cat, _path in NATURAL_FILES.items():
    _recs = list(SeqIO.parse(_path, "fasta"))
    _seqs = [str(r.seq).strip().upper() for r in _recs]
    _invalid = [r.id for r, seq in zip(_recs, _seqs) if not seq or set(seq) - set("ACDEFGHIKLMNPQRSTVWY")]
    _unique = len(set(_seqs))
    print(f"Reference {_cat}: N={len(_recs)}, unique={_unique}, invalid={len(_invalid)}; file={_path}")
    if len(_recs) != 229 or _unique != 229 or _invalid:
        raise ValueError(f"{_cat} reference failed quality controls; expected 229 unique canonical FASTA sequences.")


# Reproducibility boundary: these are PRECONSTRUCTED reference FASTA inputs.
# The validation below does not regenerate APD6 / IEDB sampling or cyclic monomer mapping.
# Publish construction scripts plus source accession/export versions and manifests separately.


Step 1/4 - Upload the FASTA file with your RL/model-generated peptides:


Saving Property-based benchmarking.fasta to Property-based benchmarking.fasta

Step 2/4 - Upload the FASTA file with selected APD6 AMP reference peptides:


Saving APD6_random_229_seed42.fasta to APD6_random_229_seed42.fasta

Step 3/4 - Upload the FASTA file with selected IEDB MHC ligand reference peptides:


Saving IEDB_random_229_seed42.fasta to IEDB_random_229_seed42.fasta

Step 4/4 - Upload the FASTA file with mapped CycPeptMPDB cyclic reference peptides:


Saving CycPeptMPDB_exact_unique.fasta to CycPeptMPDB_exact_unique.fasta
Reference AMP: N=229, unique=229, invalid=0; file=APD6_random_229_seed42.fasta
Reference MHC: N=229, unique=229, invalid=0; file=IEDB_random_229_seed42.fasta
Reference Cyclic: N=229, unique=229, invalid=0; file=CycPeptMPDB_exact_unique.fasta


In [4]:
#@title 🧹 3. Data Preparation Helpers { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-dataprep {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-dataprep {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-dataprep {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-dataprep:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-dataprep" id="headerBox_dataprep">
    <div class="title-dataprep">🧹 3. Data Preparation Helpers</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        This single cell defines every helper used to turn a raw FASTA record into a clean, labeled row:
        <ul>
        <li><b>validate_sequence</b> — removes whitespace, validates against the 20 standard amino acids, and excludes the entire peptide if any non-canonical residue is present.</li>
        <li><b>extract_info / normalize_type</b> — pulls the model name and biological <code>Category</code>
        (AMP / MHC / Cyclic) from the header.</li>
        <li><b>classify_strategy</b> (NEW) — a <i>separate</i> generation-method label
        (<code>RL</code> / <code>Pseudo-RL</code> / <code>Hybrid and Advanced</code> / <code>Other</code>), independent
        of Category, since one strategy can produce peptides across multiple categories. Fill in
        <code>STRATEGY_OVERRIDE</code> below if your naming convention doesn't contain these keywords.</li>
        <li><b>compute_properties</b> — Length, MW, GRAVY, pI, NetCharge (charge at pH 7, BioPython approximation),
        Aliphatic Index.</li>
        </ul>
    </div>
</div>
<button class="toggle-btn-dataprep" onclick="toggleHeader_dataprep()">Hide card</button>
<script>
function toggleHeader_dataprep() {
    const box = document.getElementById("headerBox_dataprep");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def validate_sequence(seq):
    """
    Remove whitespace only. If ANY non-canonical amino-acid symbol remains,
    exclude the ENTIRE peptide from quantitative benchmarking.
    """
    seq = re.sub(r"\s+", "", str(seq).upper())
    invalid = sorted(set(seq) - STANDARD_AA)
    if invalid:
        return None, invalid
    return seq, []

def standardize_header(header):
    header = header.strip()
    header = re.sub(r"\s+", " ", header)
    return header

# Audit-friendly model aliases: preserve the complete model name instead of
# taking only the first alphanumeric word in a FASTA header.
KNOWN_MODEL_NAMES = {
    "HMAMP", "DIFF-AMP", "TARSA", "ULTRAMUTATE", "OH ET AL",
    "AMP-RL", "AMPAINTER", "RR-ADS", "PEPGAN", "RLPMIEC",
    "CYC_BUILDER", "HIGHPLAY", "VAE-MH", "RF-AF3 DRL",
    "MOQA", "OH_ET_AL",
}

# Biological class is independent from generation strategy.
# Unknown cases are intentionally flagged as Mixed, not guessed.
MODEL_CATEGORY_OVERRIDE = {
    "HMAMP": "AMP", "DIFF-AMP": "AMP", "AMP-RL": "AMP",
    "AMPAINTER": "AMP", "RR-ADS": "AMP", "PEPGAN": "AMP",
    "ULTRAMUTATE": "MHC", "RLPMIEC": "MHC",
    "CYC_BUILDER": "Cyclic", "HIGHPLAY": "Cyclic",
    "TARSA": "Mixed", "OH ET AL": "Mixed", "OH_ET_AL": "Mixed",
    "MOQA": "AMP",
    "VAE-MH": "Mixed", "RF-AF3 DRL": "Mixed",
}

def extract_info(header):
    header = standardize_header(header).lstrip(">")
    type_match = re.search(r"\(([^()]*)\)", header)
    pep_type = type_match.group(1).strip() if type_match else "Unknown"
    leading = header.split("(", 1)[0].strip()
    base = re.sub(r"_\d+\s*$", "", leading).strip()
    # Strip experimental condition suffixes from AMP-RL headers, but
    # keep model identity and the remaining original header in the audit.
    if base.upper().startswith("AMP-RL_"):
        base = "AMP-RL"
    model_name = base.upper() if base else "UNKNOWN"
    return model_name, pep_type

def normalize_type(pep_type, model_name=None):
    t = re.sub(r"[\s_-]+", " ", str(pep_type).strip().lower())
    type_category = None
    if "antimicrobial" in t or re.search(r"\bamp\b", t):
        type_category = "AMP"
    elif re.search(r"\b(?:mhc|hla|phla)(?:\s*[i1])?\b|\bmhc\s*binding\b", t):
        type_category = "MHC"
    elif "cyclic" in t or "macrocyclic" in t:
        type_category = "Cyclic"
    # Strategy-only labels such as Pseudo-RL or Hybrid and Advanced
    # do not define biological category.
    mapped = MODEL_CATEGORY_OVERRIDE.get(str(model_name).upper())
    if mapped and type_category and mapped != type_category:
        raise ValueError(f"Conflicting category for {model_name}: {pep_type!r} vs {mapped}")
    return type_category or mapped or "Mixed"

# --- Generation strategy (NEW): independent of Category ---
# Explicit overrides checked first, e.g. STRATEGY_OVERRIDE = {"HMAMP": "RL", "GANHYB": "Hybrid and Advanced"}
STRATEGY_OVERRIDE = {
    # Explicit RL / policy-gradient / PPO / REINFORCE-type approaches
    "HMAMP": "RL",
    "TARSA": "RL",
    "AMP": "RL",          # AMP-RL
    "RR": "RL",           # RR-ADS
    "PEPGAN": "RL",
    "RLPMIEC": "RL",

    # Hybrid / advanced RL frameworks
    "DIFF": "Hybrid and Advanced",        # Diff-AMP: diffusion + RL
    "AMPAINTER": "Hybrid and Advanced",   # RL policy + language model
    "ULTRAMUTATE": "Hybrid and Advanced", # RL + MCTS
    "CYC": "Hybrid and Advanced",         # CYC_BUILDER: MCTS + RL
    "HIGHPLAY": "Hybrid and Advanced",    # MCTS + policy-value network

    # RL-inspired / pseudo-RL approaches
    "OH": "Pseudo-RL",     # Oh et al.
    "MOQA": "Pseudo-RL",
    "VAE": "Pseudo-RL",    # VAE-MH
    "RF": "Pseudo-RL",     # RF-AF3
}

STRATEGY_OVERRIDE.update({
    "DIFF-AMP": "Hybrid and Advanced", "AMP-RL": "RL",
    "RR-ADS": "RL", "OH ET AL": "Pseudo-RL",
    "OH_ET_AL": "Pseudo-RL",
    "VAE-MH": "Pseudo-RL", "RF-AF3 DRL": "Pseudo-RL",
    "CYC_BUILDER": "Hybrid and Advanced",
})

STRATEGY_PATTERNS = [
    ("Pseudo-RL", re.compile(r"pseudo[\s\-_]?rl", re.IGNORECASE)),
    ("Hybrid and Advanced", re.compile(r"hybrid|advanced", re.IGNORECASE)),
    ("RL", re.compile(r"\brl\b|reinforcement", re.IGNORECASE)),
]

def classify_strategy(model_name, raw_header):
    if model_name in STRATEGY_OVERRIDE:
        return STRATEGY_OVERRIDE[model_name]
    text = f"{model_name} {raw_header}"
    for label, pattern in STRATEGY_PATTERNS:
        if pattern.search(text):
            return label
    return "Other"

# --- Properties ---
PROPERTIES = ["Length", "MW", "GRAVY", "pI", "NetCharge", "AliphaticIndex"]
# Reviewer-driven applicability rule: raw sequence descriptors remain available for audit
# and descriptive plots, but are NOT a molecular-chemistry representation of cyclic peptides.
# CycPeptMPDB reference FASTA lacks molecule-specific topology, terminal caps and modifications.
# Therefore cyclic PRIMARY inferential comparisons and composite rankings use only the
# three composition-based descriptors below; no topology/capping state is invented.
CYCLIC_PRIMARY_PROPERTIES = ["Length", "GRAVY", "AliphaticIndex"]
CYCLIC_EXPLORATORY_ONLY = ["MW", "pI", "NetCharge"]
def primary_properties(category):
    return CYCLIC_PRIMARY_PROPERTIES if category == "Cyclic" else PROPERTIES


def isoelectric_point_extended(analysis, min_pH=0.0, max_pH=14.0, tolerance=1e-7, max_iterations=90):
    """Solve the SAME Biopython charge_at_pH model without its usual 4.05--12 pH bounds.

    A non-bracketed sequence is NEVER assigned a fabricated boundary pI: raise an error
    so the user can inspect the record before any downstream analyses are run.
    """
    charge_low = float(analysis.charge_at_pH(min_pH))
    charge_high = float(analysis.charge_at_pH(max_pH))
    if not (np.isfinite(charge_low) and np.isfinite(charge_high)):
        raise ValueError("Non-finite charge in extended pI calculation")
    if abs(charge_low) <= 1e-12:
        return float(min_pH)
    if abs(charge_high) <= 1e-12:
        return float(max_pH)
    if charge_low * charge_high > 0:
        raise ValueError(f"pI not bracketed in [{min_pH}, {max_pH}]; "
                         f"charges={charge_low:.6g}, {charge_high:.6g}")
    lo, hi = float(min_pH), float(max_pH)
    for _ in range(max_iterations):
        mid = (lo + hi) / 2.0
        charge_mid = float(analysis.charge_at_pH(mid))
        if not np.isfinite(charge_mid):
            raise ValueError("Non-finite intermediate charge in extended pI")
        if abs(charge_mid) <= 1e-10 or (hi - lo) <= tolerance:
            return mid
        if charge_low * charge_mid > 0:
            lo, charge_low = mid, charge_mid
        else:
            hi = mid
    return (lo + hi) / 2.0


def compute_properties(seq):
    if len(seq) == 0:
        return None
    analysis = ProteinAnalysis(seq)
    return {
        "Length": len(seq),
        "MW": analysis.molecular_weight(),
        "GRAVY": analysis.gravy(),
        "pI": isoelectric_point_extended(analysis),
        "NetCharge": analysis.charge_at_pH(7.0),
        "AliphaticIndex": (
            (seq.count("A") +
             2.9 * seq.count("V") +
             3.9 * seq.count("I") +
             3.9 * seq.count("L")) * 100 / len(seq)
        )
    }


In [5]:
#@title 📄 4. Parse FASTA Files → Build Dataset { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-parse {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-parse {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-parse {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-parse:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-parse" id="headerBox_parse">
    <div class="title-parse">📄 4. Parse FASTA Files → Build Dataset</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Parses the main (generated) file and all three natural files into one combined DataFrame <code>df</code>,
        tagging every row with <code>Model</code>, <code>Category</code>, and (for generated peptides)
        <code>Strategy</code>. Also prints diagnostics: unparsed headers, models that fell back to
        <code>Other</code> strategy, and (Category, Model) groups too small (&lt;3) for statistical testing later.
    </div>
</div>
<button class="toggle-btn-parse" onclick="toggleHeader_parse()">Hide card</button>
<script>
function toggleHeader_parse() {
    const box = document.getElementById("headerBox_parse");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


rows = []
excluded_rows = []

# Generated/model peptides
for record in SeqIO.parse(main_fasta_file, "fasta"):
    raw_seq = str(record.seq)
    model_name, pep_type = extract_info(record.description)
    category = normalize_type(pep_type, model_name)
    strategy = classify_strategy(model_name, record.description)

    valid_seq, invalid = validate_sequence(raw_seq)

    # IMPORTANT: exclude the whole peptide if any invalid/non-canonical residue exists
    if valid_seq is None:
        excluded_rows.append({
            "ID": record.id,
            "Source": "Generated",
            "Model": model_name,
            "Category": category,
            "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
            "InvalidAA": ",".join(invalid),
            "Reason": "Excluded: contains non-canonical amino-acid symbol(s)"
        })
        continue

    props = compute_properties(valid_seq)
    if props is None:
        excluded_rows.append({
            "ID": record.id,
            "Source": "Generated",
            "Model": model_name,
            "Category": category,
            "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
            "InvalidAA": None,
            "Reason": "Excluded: empty/invalid sequence"
        })
        continue

    rows.append({
        "ID": record.id,
        "Model": model_name,
        "Strategy": strategy,
        "RawType": pep_type,
        "Category": category,
        "CleanSeq": valid_seq,
        "InvalidAA": None,
        **props
    })

# Selected database reference peptides
for category, fasta_path in NATURAL_FILES.items():
    for record in SeqIO.parse(fasta_path, "fasta"):
        raw_seq = str(record.seq)
        valid_seq, invalid = validate_sequence(raw_seq)

        if valid_seq is None:
            excluded_rows.append({
                "ID": record.id,
                "Source": f"Natural {category}",
                "Model": "Natural",
                "Category": category,
                "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
                "InvalidAA": ",".join(invalid),
                "Reason": "Excluded: contains non-canonical amino-acid symbol(s)"
            })
            continue

        props = compute_properties(valid_seq)
        if props is None:
            excluded_rows.append({
                "ID": record.id,
                "Source": f"Natural {category}",
                "Model": "Natural",
                "Category": category,
                "RawSequence": re.sub(r"\\s+", "", raw_seq.upper()),
                "InvalidAA": None,
                "Reason": "Excluded: empty/invalid sequence"
            })
            continue

        rows.append({
            "ID": record.id,
            "Model": "Natural",  # internal bookkeeping key; plots use reference label
            "Strategy": "Natural",
            "RawType": f"Natural {category}",
            "Category": category,
            "CleanSeq": valid_seq,
            "InvalidAA": None,
            **props
        })

df = pd.DataFrame(rows)
excluded_df = pd.DataFrame(excluded_rows)

# Classification audit: run before interpreting comparative rankings.
_generated_audit = (df[df["Model"] != "Natural"]
    .groupby(["Model", "RawType", "Category", "Strategy"], dropna=False)
    .size().reset_index(name="N_sequences")
    .sort_values(["Category", "Model"]))
_expected_categories = {
    "HMAMP":"AMP", "DIFF-AMP":"AMP", "AMP-RL":"AMP", "AMPAINTER":"AMP",
    "RR-ADS":"AMP", "PEPGAN":"AMP", "ULTRAMUTATE":"MHC", "RLPMIEC":"MHC",
    "CYC_BUILDER":"Cyclic", "HIGHPLAY":"Cyclic", "MOQA":"AMP", "TARSA":"Mixed",
    "OH ET AL":"Mixed", "OH_ET_AL":"Mixed", "VAE-MH":"Mixed", "RF-AF3 DRL":"Mixed"
}
_generated_check = df[df["Model"] != "Natural"]
for _model, _expected in _expected_categories.items():
    _found = set(_generated_check.loc[_generated_check["Model"] == _model, "Category"])
    if _found and _found != {_expected}:
        raise AssertionError(f"Classification mismatch: {_model}: {_found} vs {_expected}")
_unknown_models = set(_generated_check["Model"]) - set(_expected_categories)
if _unknown_models:
    raise ValueError(f"Unrecognized models: {sorted(_unknown_models)}. Explicit review needed before analysis.")
print("\n=== Generated peptide model/category/strategy audit ===")
display(_generated_audit)
_generated_audit.to_csv("generated_header_category_audit.csv", index=False)
_unresolved = _generated_audit[_generated_audit["Category"] == "Mixed"]
if len(_unresolved):
    print("\nREVIEW NEEDED: these papers lack a confirmed biological category; ")
    print("they are retained as Mixed instead of being assigned by a strategy label.")
    display(_unresolved)



print(f"Total INCLUDED sequences: {len(df)}")
print(f"Total EXCLUDED sequences: {len(excluded_df)}")

if len(excluded_df) > 0:
    print("\nExcluded sequences:")
    display(excluded_df)
    excluded_df.to_csv("excluded_sequences.csv", index=False)
    print("\nSaved exclusion log: excluded_sequences.csv")
print(f"Total sequences: {len(df)} | Models: {list(df['Model'].unique())} | "
      f"Strategies: {list(df['Strategy'].unique())} | Categories: {list(df['Category'].unique())}")

unresolved = df[(df["Model"] == "UNKNOWN") | (df["RawType"] == "Unknown")]
unresolved = unresolved[unresolved["Model"] != "Natural"]
if len(unresolved) > 0:
    print(f"\nWARNING: {len(unresolved)} sequence(s) could not be fully parsed. "
          f"Expected header format: '>ModelName_ID (Type)'. Examples:")
    print(unresolved[["ID", "Model", "RawType", "Category"]].head(10).to_string(index=False))

other_models = sorted(df[(df["Strategy"] == "Other")]["Model"].unique())
if other_models:
    print(f"\nNOTE: models labeled 'Other' strategy (not matched by keyword/override): {other_models}. "
          f"Add them to STRATEGY_OVERRIDE above if needed.")

small_models = df[df["Model"] != "Natural"].groupby(["Category", "Model"]).size()
small_models = small_models[small_models < 3]
if len(small_models) > 0:
    print(f"\nWARNING: these (Category, Model) groups have < 3 sequences and will be skipped "
          f"in the statistical comparison:")
    print(small_models.to_string())



=== Generated peptide model/category/strategy audit ===


,Model,RawType,Category,Strategy,N_sequences
0,AMP-RL,antimicrobial peptide,AMP,RL,6
1,AMPAINTER,antimicrobial peptide,AMP,Hybrid and Advanced,30
3,DIFF-AMP,antimicrobial peptide,AMP,Hybrid and Advanced,6
5,HMAMP,Unknown,AMP,RL,10
6,MOQA,AMP; Pseudo-RL,AMP,Pseudo-RL,4
8,PEPGAN,antimicrobial peptide,AMP,RL,6
11,RR-ADS,antimicrobial peptide,AMP,RL,15
2,CYC_BUILDER,Cyclic and Macrocyclic,Cyclic,Hybrid and Advanced,9
4,HIGHPLAY,Cyclic and Macrocyclic,Cyclic,Hybrid and Advanced,30
10,RLPMIEC,MHC-Binding,MHC,RL,2



REVIEW NEEDED: these papers lack a confirmed biological category; 
they are retained as Mixed instead of being assigned by a strategy label.


,Model,RawType,Category,Strategy,N_sequences
7,OH_ET_AL,Pseudo-RL,Mixed,Pseudo-RL,10
9,RF-AF3 DRL,Pseudo-RL,Mixed,Pseudo-RL,1
12,TARSA,Hybrid and Advanced,Mixed,RL,3
14,VAE-MH,Pseudo-RL,Mixed,Pseudo-RL,15


Total INCLUDED sequences: 836
Total EXCLUDED sequences: 0
Total sequences: 836 | Models: ['HMAMP', 'DIFF-AMP', 'TARSA', 'ULTRAMUTATE', 'OH_ET_AL', 'AMP-RL', 'AMPAINTER', 'RR-ADS', 'PEPGAN', 'RLPMIEC', 'CYC_BUILDER', 'HIGHPLAY', 'VAE-MH', 'RF-AF3 DRL', 'MOQA', 'Natural'] | Strategies: ['RL', 'Hybrid and Advanced', 'Pseudo-RL', 'Natural'] | Categories: ['AMP', 'Mixed', 'MHC', 'Cyclic']

      ID Model RawType Category
 HMAMP_1 HMAMP Unknown      AMP
 HMAMP_2 HMAMP Unknown      AMP
 HMAMP_3 HMAMP Unknown      AMP
 HMAMP_4 HMAMP Unknown      AMP
 HMAMP_5 HMAMP Unknown      AMP
 HMAMP_6 HMAMP Unknown      AMP
 HMAMP_7 HMAMP Unknown      AMP
 HMAMP_8 HMAMP Unknown      AMP
 HMAMP_9 HMAMP Unknown      AMP
HMAMP_10 HMAMP Unknown      AMP

Category  Model      
MHC       RLPMIEC        2
          ULTRAMUTATE    2
Mixed     RF-AF3 DRL     1


In [6]:
#@title 🧾 4B. pI validity and FASTA record-retention audit
# Run immediately after Cell 4, before any statistical comparisons.
# This checks BOTH pI validity and retention relative to the original FASTA records.
import numpy as np
import pandas as pd
from Bio import SeqIO

original_inputs = [("Generated", main_fasta_file)] + [
    (f"Reference_{category}", fasta_path) for category, fasta_path in NATURAL_FILES.items()
]
input_counts = {label: sum(1 for _ in SeqIO.parse(path, "fasta"))
                for label, path in original_inputs}
input_total = sum(input_counts.values())
included_total = len(df)
excluded_total = len(excluded_df)

# Each parsed FASTA record must lead to an included row or an explicitly logged exclusion.
assert input_total == included_total + excluded_total, (
    f"FASTA retention mismatch: {input_total} raw records, "
    f"{included_total} included and {excluded_total} logged excluded."
)
assert "pI" in df.columns, "pI property not present in final dataframe."
pi_numeric = pd.to_numeric(df["pI"], errors="coerce").to_numpy(dtype=float)
invalid_pi_mask = ~np.isfinite(pi_numeric)
invalid_pi_count = int(invalid_pi_mask.sum())

# Distinguish allowed, documented sequence exclusions from forbidden silent pI exclusions.
if excluded_total:
    assert "Reason" in excluded_df.columns, "Exclusions must have a Reason."
    reasons = excluded_df["Reason"].fillna("").astype(str)
    assert reasons.str.strip().ne("").all(), "Unexplained exclusions found."
    pi_related = reasons.str.contains(r"\bpI\b|isoelectric|charge.*root|bracket|non.finite", case=False, regex=True)
    assert not pi_related.any(), (
        "Some FASTA input sequences were excluded due to pI/root calculation: "
        + excluded_df.loc[pi_related, ["ID", "Reason"]].to_string(index=False)
    )

pi_status = df[["ID", "Category", "Model", "CleanSeq", "pI"]].copy()
pi_status["pI_finite"] = ~invalid_pi_mask
pi_status.to_csv("pI_record_retention_audit.csv", index=False)

print("=== FASTA input and pI retention audit ===")
print("Input FASTA records:", input_counts)
print("Raw input total:", input_total)
print("Included:", included_total)
print("Logged exclusions (sequence eligibility):", excluded_total)
print("Non-finite pI among included records:", invalid_pi_count)
print("Extended pI range among included records:",
      (float(np.min(pi_numeric)), float(np.max(pi_numeric))) if included_total and not invalid_pi_count else "not available")

assert invalid_pi_count == 0, (
    f"{invalid_pi_count} included sequence(s) have non-finite pI values; "
    "investigate before statistical analysis."
)
assert included_total > 0, "No valid sequence records were loaded."
print("PASS: all input records accounted for; no pI-related omissions; all included pI values finite.")
print("Saved: pI_record_retention_audit.csv")


=== FASTA input and pI retention audit ===
Input FASTA records: {'Generated': 149, 'Reference_AMP': 229, 'Reference_MHC': 229, 'Reference_Cyclic': 229}
Raw input total: 836
Included: 836
Logged exclusions (sequence eligibility): 0
Non-finite pI among included records: 0
Extended pI range among included records: (3.321258556097746, 13.00000149384141)
PASS: all input records accounted for; no pI-related omissions; all included pI values finite.
Saved: pI_record_retention_audit.csv


In [7]:
#@title 🔎 4A. FASTA Input Integrity, Duplicate, Overlap & Similarity Audit
# Run after parsing (Cell 4), before baseline analysis. Audits are diagnostic: no rows are removed.
from itertools import combinations
from collections import Counter

_audit = df[["ID", "Category", "Model", "CleanSeq"]].copy().reset_index(drop=True)
_audit["Population"] = np.where(_audit["Model"].eq("Natural"), "Reference", "Generated")
_audit["Sequence_normalized"] = _audit["CleanSeq"].astype(str).str.upper().str.replace(r"\s+", "", regex=True)
_audit["Group"] = _audit["Population"] + ":" + _audit["Category"].astype(str) + ":" + _audit["Model"].astype(str)

# Exact duplicates WITHIN a model/reference set, retained in audit output.
_audit["Within_group_copy_count"] = _audit.groupby(["Group","Sequence_normalized"])["ID"].transform("size")
within_duplicate_df = _audit[_audit.Within_group_copy_count > 1][
    ["Population", "Category", "Model", "ID", "Sequence_normalized", "Within_group_copy_count"]].copy()

# Pairwise exact matches BETWEEN groups. Detect different models, generated-v-reference,
# and cross-category references (including reference-reference overlap).
exact_overlap_rows = []
for sequence, block in _audit.groupby("Sequence_normalized", sort=False):
    distinct_groups = block["Group"].nunique()
    if distinct_groups < 2: continue
    entries = block[["ID", "Population", "Category", "Model", "Group"]].to_dict("records")
    for left, right in combinations(entries, 2):
        if left["Group"] == right["Group"]: continue
        exact_overlap_rows.append({
            "Sequence":sequence, "Left_ID":left["ID"], "Left_Population":left["Population"],
            "Left_Category":left["Category"], "Left_Model":left["Model"],
            "Right_ID":right["ID"], "Right_Population":right["Population"],
            "Right_Category":right["Category"], "Right_Model":right["Model"],
            "Comparison": ("Generated vs reference" if left["Population"] != right["Population"]
                else "Between generated models" if left["Population"] == "Generated" else "Between reference subsets")})
exact_overlap_df = pd.DataFrame(exact_overlap_rows, columns=["Sequence", "Left_ID", "Left_Population",
    "Left_Category", "Left_Model", "Right_ID", "Right_Population", "Right_Category", "Right_Model", "Comparison"])

# Pairwise normalized Levenshtein similarity WITHIN each generated model.
# Similarity=1 - edit_distance/max(sequence lengths); descriptive flag >=0.80 only.
# This is not an independence test. It does not claim evolutionary homology.
def _levenshtein_distance(x, y):
    if len(x) < len(y): x, y = y, x
    prev = list(range(len(y)+1))
    for i, char_a in enumerate(x, 1):
        cur = [i]
        for j, char_b in enumerate(y, 1):
            cur.append(min(cur[j-1]+1, prev[j]+1, prev[j-1]+int(char_a != char_b)))
        prev = cur
    return prev[-1]

near_pairs = []
for (cat, model), block in _audit[_audit.Population == "Generated"].groupby(["Category", "Model"]):
    recs = block[["ID", "Sequence_normalized"]].to_dict("records")
    for rec_a, rec_b in combinations(recs, 2):
        sa, sb = rec_a["Sequence_normalized"], rec_b["Sequence_normalized"]
        distance = _levenshtein_distance(sa, sb)
        similarity = 1 - distance/max(len(sa), len(sb))
        if similarity >= 0.70:
            near_pairs.append({"Category":cat,"Model":model,"ID_1":rec_a["ID"],"ID_2":rec_b["ID"],
                "Sequence_1":sa,"Sequence_2":sb,"Edit_distance":distance,
                "Normalized_edit_similarity":round(similarity,4)})
near_similarity_all_df = pd.DataFrame(near_pairs, columns=["Category","Model","ID_1","ID_2",
    "Sequence_1","Sequence_2","Edit_distance","Normalized_edit_similarity"])
near_similarity_df = near_similarity_all_df[near_similarity_all_df["Normalized_edit_similarity"] >= 0.80].copy()
near_similarity_070_df = near_similarity_all_df.copy()
# 0.70/0.80 are exploratory flags, not validated sequence-dependence thresholds.


# Audit ALL INPUT FASTA ROWS (before validity exclusion) vs retained counts.
input_audit_rows=[]
for population, category, file_path in [("Generated", "All", main_fasta_file)] + [
    ("Reference", cat, path) for cat,path in NATURAL_FILES.items()]:
    original = list(SeqIO.parse(file_path, "fasta"))
    input_audit_rows.append({"Population":population,"Category":category,
        "Input_records":len(original),
        "Retained_records":int((_audit.Population.eq(population) &
                                   ((_audit.Category.eq(category)) if category != "All" else True)).sum()),
        "Input_unique_normalized_sequences":len(set(re.sub(r"\s+", "", str(rec.seq).upper()) for rec in original)),
        "Invalid_excluded":sum(1 for rec in original if validate_sequence(str(rec.seq))[0] is None)})
input_integrity_df = pd.DataFrame(input_audit_rows)

duplicate_summary_rows=[]
for (population,category,model), block in _audit.groupby(["Population","Category","Model"]):
    duplicate_summary_rows.append({"Population":population,"Category":category,"Model":model,
        "N_sequences":len(block),"Exact_duplicate_extra_copies":int(len(block)-block.Sequence_normalized.nunique()),
        "Within_model_high_similarity_pairs_ge_0_80": int(((near_similarity_df.Category==category) &
          (near_similarity_df.Model==model)).sum()) if population=="Generated" else np.nan})
duplicate_summary_df=pd.DataFrame(duplicate_summary_rows).sort_values(["Population","Category","Model"])

for _name, _tbl in [("input_integrity_audit",input_integrity_df),
                    ("duplicate_summary_audit",duplicate_summary_df),
                    ("within_group_exact_duplicates",within_duplicate_df),
                    ("cross_group_exact_overlap",exact_overlap_df),
                    ("within_model_high_similarity_pairs",near_similarity_df),
                    ("within_model_similarity_pairs_ge_0_70",near_similarity_070_df)]:
    _tbl.to_csv(_name+".csv", index=False)

print("=== Raw input vs retained FASTA records ===")
display(input_integrity_df)
print("=== Group-level exact duplication and near-similarity diagnostics ===")
display(duplicate_summary_df)
print("Exact duplicate rows within groups:",len(within_duplicate_df),
      "| Cross-group exact overlap pairs:",len(exact_overlap_df),
      "| Within-model similarity >=0.80 pairs:",len(near_similarity_df),
      "| >=0.70 pairs:",len(near_similarity_070_df))
if len(exact_overlap_df): display(exact_overlap_df.head(30))
if len(near_similarity_df): display(near_similarity_df.head(30))
print("Note: Exact uniqueness DOES NOT imply independence. High edit similarity is an exploratory flag, not proof of dependence.")


=== Raw input vs retained FASTA records ===


,Population,Category,Input_records,Retained_records,Input_unique_normalized_sequences,Invalid_excluded
0,Generated,All,149,149,149,0
1,Reference,AMP,229,229,229,0
2,Reference,MHC,229,229,229,0
3,Reference,Cyclic,229,229,229,0


=== Group-level exact duplication and near-similarity diagnostics ===


,Population,Category,Model,N_sequences,Exact_duplicate_extra_copies,Within_model_high_similarity_pairs_ge_0_80
0,Generated,AMP,AMP-RL,6,0,0.0
1,Generated,AMP,AMPAINTER,30,0,1.0
2,Generated,AMP,DIFF-AMP,6,0,0.0
3,Generated,AMP,HMAMP,10,0,0.0
4,Generated,AMP,MOQA,4,0,0.0
5,Generated,AMP,PEPGAN,6,0,0.0
6,Generated,AMP,RR-ADS,15,0,0.0
7,Generated,Cyclic,CYC_BUILDER,9,0,2.0
8,Generated,Cyclic,HIGHPLAY,30,0,26.0
9,Generated,MHC,RLPMIEC,2,0,0.0


Exact duplicate rows within groups: 0 | Cross-group exact overlap pairs: 0 | Within-model similarity >=0.80 pairs: 35 | >=0.70 pairs: 57


,Category,Model,ID_1,ID_2,Sequence_1,Sequence_2,Edit_distance,Normalized_edit_similarity
1,AMP,AMPAINTER,AMPainter_5,AMPainter_7,GIKKWIKGWAKKLIKKVAKKIAKQ,GIKKWIKGAIKKLIKKVAKWIWKQ,4,0.8333
3,Cyclic,CYC_BUILDER,CYC_BUILDER_3,CYC_BUILDER_8,AYLVLELNDPL,AYLVLELNDSK,2,0.8182
7,Cyclic,CYC_BUILDER,CYC_BUILDER_8,CYC_BUILDER_9,AYLVLELNDSK,AYLVEELNDSQ,2,0.8182
8,Cyclic,HIGHPLAY,HighPlay_1,HighPlay_2,TPCYMEGENDNDCK,TPCYMEGANDNDCK,1,0.9286
9,Cyclic,HIGHPLAY,HighPlay_1,HighPlay_3,TPCYMEGENDNDCK,TPCYMENENDNDCK,1,0.9286
10,Cyclic,HIGHPLAY,HighPlay_1,HighPlay_4,TPCYMEGENDNDCK,TPCYMENANDNDCK,2,0.8571
12,Cyclic,HIGHPLAY,HighPlay_2,HighPlay_3,TPCYMEGANDNDCK,TPCYMENENDNDCK,2,0.8571
13,Cyclic,HIGHPLAY,HighPlay_2,HighPlay_4,TPCYMEGANDNDCK,TPCYMENANDNDCK,1,0.9286
14,Cyclic,HIGHPLAY,HighPlay_2,HighPlay_5,TPCYMEGANDNDCK,TPCYMENARDNDCK,2,0.8571
15,Cyclic,HIGHPLAY,HighPlay_3,HighPlay_4,TPCYMENENDNDCK,TPCYMENANDNDCK,1,0.9286


Note: Exact uniqueness DOES NOT imply independence. High edit similarity is an exploratory flag, not proof of dependence.


In [9]:

#@title 🧬 4C. Local Sequence Similarity to Selected References — Calibrated, Ring-Aware
# INDEPENDENT: no mutation of df, pI, Wasserstein, bootstrap, LOFO, or previous audit results.
# Biopython Smith-Waterman (LOCAL, BLOSUM62); NOT BLAST or CD-HIT.
# Selected same-category subsets only (229 per category), NOT all database/training records.

from Bio.Align import PairwiseAligner, substitution_matrices
from IPython.display import display
import numpy as np
import pandas as pd

assert "df" in globals() and {"Category", "Model", "ID", "CleanSeq"}.issubset(df.columns)

SIM_SEED = 42
SIM_MIN_QUERY_COVERAGE = 0.70
SIM_MIN_ALIGNED_AA = 5
SIM_SHUFFLES_PER_GENERATED = 1  # one composition-preserving shuffle; exploratory null baseline

_rng_sim = np.random.default_rng(SIM_SEED)

_sw = PairwiseAligner()
_sw.mode = "local"
_sw.substitution_matrix = substitution_matrices.load("BLOSUM62")
_sw.open_gap_score = -10
_sw.extend_gap_score = -0.5

_sim_generated = df.loc[
    df.Model != "Natural",
    ["ID", "Category", "Model", "CleanSeq"]
].copy()

_sim_ref = df.loc[
    df.Model == "Natural",
    ["ID", "Category", "CleanSeq"]
].copy()

_sim_refs = {
    cat: sorted([
        (str(t.ID), str(t.CleanSeq).upper())
        for t in grp.itertuples(index=False)
    ])
    for cat, grp in _sim_ref.groupby("Category")
}


def _cyclic_query_rotations(seq, model):
    # Only documented head-to-tail CYC_BUILDER: start-position invariance.
    return (
        sorted(set(seq[i:] + seq[:i] for i in range(len(seq))))
        if model == "CYC_BUILDER"
        else [seq]
    )


def _local_match(query, reference):
    # Report identity over aligned residue pairs (not BLAST identity);
    # coverage is calculated over the full query.

    alignments = _sw.align(query, reference)

    # FIX: Smith-Waterman may return no positive-scoring alignment.
    # In that case, do not access alignments[0].
    if len(alignments) == 0:
        return (np.nan, 0.0, 0, 0.0)

    aln = alignments[0]
    ai, bi = aln.indices

    paired = (ai >= 0) & (bi >= 0)
    npairs = int(paired.sum())

    if npairs == 0:
        return (np.nan, 0.0, 0, float(aln.score))

    matches = sum(
        query[int(a)] == reference[int(b)]
        for a, b in zip(ai[paired], bi[paired])
    )

    coverage = float(
        len(set(int(x) for x in ai[paired])) / len(query)
    )

    return (
        float(matches / npairs),
        coverage,
        npairs,
        float(aln.score)
    )


def _best_reference(query, reference_records, model, exclude_index=None):
    best = None
    rotations = _cyclic_query_rotations(query, model)

    for j, (rid, refseq) in enumerate(reference_records):
        if j == exclude_index:
            continue

        for rotation in rotations:
            identity, coverage, length, score = _local_match(rotation, refseq)

            if length < SIM_MIN_ALIGNED_AA or coverage < SIM_MIN_QUERY_COVERAGE:
                continue

            # Prioritize maximum local identity after fixed coverage/length filters.
            key = (identity, coverage, length, score)

            if best is None or key > best[0]:
                best = (key, rid, refseq)

    return best


def _row_result(qid, cat, model, seq, records, comparison, exclude_index=None):
    if not records:
        return {
            "Category": cat,
            "Model": model,
            "Query_ID": qid,
            "Comparison": comparison,
            "N_reference": 0,
            "Nearest_reference_ID": pd.NA,
            "Local_identity": np.nan,
            "Query_coverage": np.nan,
            "Aligned_residues": 0,
            "Exact_full_sequence_match": pd.NA,
            "Status": "NO_CATEGORY_REFERENCE"
        }

    best = _best_reference(seq, records, model, exclude_index)

    if best is None:
        ident, cov, alen, rid, status = (
            np.nan, np.nan, 0, pd.NA, "NO_QUALIFYING_ALIGNMENT"
        )
    else:
        key, rid, _ = best
        ident, cov, alen = key[:3]
        status = "EVALUATED"

    # Exact match is independently checked;
    # no minimum coverage or local alignment filter.
    if model == "CYC_BUILDER":
        variants = set(_cyclic_query_rotations(seq, model))
        exact = any(
            refseq in variants
            for j, (_, refseq) in enumerate(records)
            if j != exclude_index
        )
    else:
        exact = any(
            refseq == seq
            for j, (_, refseq) in enumerate(records)
            if j != exclude_index
        )

    return {
        "Category": cat,
        "Model": model,
        "Query_ID": qid,
        "Comparison": comparison,
        "N_reference": len(records) - (exclude_index is not None),
        "Nearest_reference_ID": rid,
        "Local_identity": ident,
        "Query_coverage": cov,
        "Aligned_residues": alen,
        "Exact_full_sequence_match": bool(exact),
        "Status": status
    }


_gen_rows, _shuffle_rows, _loo_rows = [], [], []

for r in _sim_generated.itertuples(index=False):
    s = str(r.CleanSeq).upper()
    refs = _sim_refs.get(r.Category, [])

    _gen_rows.append(
        _row_result(
            str(r.ID), r.Category, r.Model, s, refs, "GENERATED"
        )
    )

    if refs:
        for k in range(SIM_SHUFFLES_PER_GENERATED):
            shuffled = "".join(_rng_sim.permutation(list(s)))

            _shuffle_rows.append(
                _row_result(
                    f"{r.ID}:shuffle{k+1}",
                    r.Category,
                    r.Model,
                    shuffled,
                    refs,
                    "SHUFFLED_GENERATED"
                )
            )

for cat, records in _sim_refs.items():
    for j, (rid, seq) in enumerate(records):
        _loo_rows.append(
            _row_result(
                rid,
                cat,
                "REFERENCE_LOO",
                seq,
                records,
                "REFERENCE_LEAVE_ONE_OUT",
                exclude_index=j
            )
        )


reference_sequence_similarity_df = pd.DataFrame(_gen_rows)
reference_similarity_shuffled_df = pd.DataFrame(_shuffle_rows)
reference_similarity_reference_loo_df = pd.DataFrame(_loo_rows)

assert len(reference_sequence_similarity_df) == len(_sim_generated)


# Report denominators explicitly;
# missing qualified alignments are NEVER interpreted as novel.
def _group_summary(tab, group_cols):
    rows = []

    if tab.empty:
        return pd.DataFrame()

    for key, g in tab.groupby(group_cols, sort=True):
        if not isinstance(key, tuple):
            key = (key,)

        valid = g[g.Status == "EVALUATED"]
        nonmissing = g[g.Status != "NO_CATEGORY_REFERENCE"]

        row = dict(zip(group_cols, key))

        row.update({
            "N_queries": len(g),
            "N_qualified_alignments": len(valid),
            "N_unqualified": int(
                (g.Status == "NO_QUALIFYING_ALIGNMENT").sum()
            ),
            "Median_max_local_identity": valid.Local_identity.median(),
            "Median_query_coverage": valid.Query_coverage.median(),
            "Full_sequence_exact_match_N": int(
                nonmissing.Exact_full_sequence_match.eq(True).sum()
            ),
            "Full_sequence_exact_match_%": (
                100 * nonmissing.Exact_full_sequence_match.eq(True).mean()
                if len(nonmissing) else np.nan
            ),
            "Local_identity_ge_0_80_N": int(
                (valid.Local_identity >= .80).sum()
            ),
            "Local_identity_ge_0_90_N": int(
                (valid.Local_identity >= .90).sum()
            ),
            "Local_identity_ge_0_80_%_of_all": (
                100 * (valid.Local_identity >= .80).sum() / len(g)
            ),
            "Local_identity_ge_0_90_%_of_all": (
                100 * (valid.Local_identity >= .90).sum() / len(g)
            )
        })

        rows.append(row)

    return pd.DataFrame(rows)


reference_sequence_similarity_summary_df = _group_summary(
    reference_sequence_similarity_df,
    ["Category", "Model"]
)

reference_similarity_shuffled_summary_df = _group_summary(
    reference_similarity_shuffled_df,
    ["Category", "Model"]
)

reference_similarity_loo_summary_df = _group_summary(
    reference_similarity_reference_loo_df,
    ["Category"]
)


for name, tab in [
    (
        "generated_vs_selected_reference_local_similarity.csv",
        reference_sequence_similarity_df
    ),
    (
        "generated_vs_selected_reference_local_similarity_by_model.csv",
        reference_sequence_similarity_summary_df
    ),
    (
        "composition_shuffled_generated_local_similarity.csv",
        reference_similarity_shuffled_df
    ),
    (
        "composition_shuffled_local_similarity_by_model.csv",
        reference_similarity_shuffled_summary_df
    ),
    (
        "selected_reference_leave_one_out_local_similarity.csv",
        reference_similarity_reference_loo_df
    ),
    (
        "selected_reference_leave_one_out_local_similarity_by_category.csv",
        reference_similarity_loo_summary_df
    ),
]:
    tab.to_csv(name, index=False)


print(
    "Local Smith-Waterman BLOSUM62; gap open -10, extend -0.5; "
    f"min query coverage {SIM_MIN_QUERY_COVERAGE}, "
    f"min aligned pairs {SIM_MIN_ALIGNED_AA}."
)

print(
    "Generated:", len(reference_sequence_similarity_df),
    "| shuffled controls:", len(reference_similarity_shuffled_df),
    "| reference LOO:", len(reference_similarity_reference_loo_df)
)

print(
    "CYC_BUILDER only: evaluate all rotations of head-to-tail candidate sequences."
)

print(
    "BLAST/CD-HIT were not run. Best local alignment identity is NOT "
    "a global similarity or motif novelty proof."
)

print(
    "No training-set memorization inference; comparisons limited "
    "to selected same-category references."
)

print(
    "Reference LOO and shuffled controls describe baseline similarities, "
    "not formal p-values."
)

display(reference_sequence_similarity_summary_df)
display(reference_similarity_shuffled_summary_df)
display(reference_similarity_loo_summary_df)


Local Smith-Waterman BLOSUM62; gap open -10, extend -0.5; min query coverage 0.7, min aligned pairs 5.
Generated: 149 | shuffled controls: 120 | reference LOO: 687
CYC_BUILDER only: evaluate all rotations of head-to-tail candidate sequences.
BLAST/CD-HIT were not run. Best local alignment identity is NOT a global similarity or motif novelty proof.
No training-set memorization inference; comparisons limited to selected same-category references.
Reference LOO and shuffled controls describe baseline similarities, not formal p-values.


,Category,Model,N_queries,N_qualified_alignments,N_unqualified,Median_max_local_identity,Median_query_coverage,Full_sequence_exact_match_N,Full_sequence_exact_match_%,Local_identity_ge_0_80_N,Local_identity_ge_0_90_N,Local_identity_ge_0_80_%_of_all,Local_identity_ge_0_90_%_of_all
0,AMP,AMP-RL,6,6,0,0.500000,0.801282,0,0.0,0,0,0.000000,0.0
1,AMP,AMPAINTER,30,29,1,0.473684,0.764706,0,0.0,0,0,0.000000,0.0
2,AMP,DIFF-AMP,6,6,0,0.607143,0.777778,0,0.0,1,0,16.666667,0.0
3,AMP,HMAMP,10,10,0,0.445055,0.792735,0,0.0,0,0,0.000000,0.0
4,AMP,MOQA,4,4,0,0.556373,0.848485,0,0.0,0,0,0.000000,0.0
5,AMP,PEPGAN,6,6,0,0.633929,0.750000,0,0.0,0,0,0.000000,0.0
6,AMP,RR-ADS,15,15,0,0.583333,0.764706,0,0.0,0,0,0.000000,0.0
7,Cyclic,CYC_BUILDER,9,3,6,0.400000,0.818182,0,0.0,0,0,0.000000,0.0
8,Cyclic,HIGHPLAY,30,0,30,NaN,NaN,0,0.0,0,0,0.000000,0.0
9,MHC,RLPMIEC,2,2,0,0.526786,0.833333,0,0.0,0,0,0.000000,0.0


,Category,Model,N_queries,N_qualified_alignments,N_unqualified,Median_max_local_identity,Median_query_coverage,Full_sequence_exact_match_N,Full_sequence_exact_match_%,Local_identity_ge_0_80_N,Local_identity_ge_0_90_N,Local_identity_ge_0_80_%_of_all,Local_identity_ge_0_90_%_of_all
0,AMP,AMP-RL,6,6,0,0.477273,0.795977,0,0.0,0,0,0.0,0.0
1,AMP,AMPAINTER,30,26,4,0.470175,0.775253,0,0.0,0,0,0.0,0.0
2,AMP,DIFF-AMP,6,6,0,0.554945,0.777778,0,0.0,0,0,0.0,0.0
3,AMP,HMAMP,10,9,1,0.416667,0.777778,0,0.0,0,0,0.0,0.0
4,AMP,MOQA,4,4,0,0.485294,0.791126,0,0.0,0,0,0.0,0.0
5,AMP,PEPGAN,6,6,0,0.485294,0.819737,0,0.0,0,0,0.0,0.0
6,AMP,RR-ADS,15,14,1,0.500000,0.775210,0,0.0,0,0,0.0,0.0
7,Cyclic,CYC_BUILDER,9,5,4,0.444444,0.727273,0,0.0,0,0,0.0,0.0
8,Cyclic,HIGHPLAY,30,0,30,NaN,NaN,0,0.0,0,0,0.0,0.0
9,MHC,RLPMIEC,2,2,0,0.473214,0.833333,0,0.0,0,0,0.0,0.0


,Category,N_queries,N_qualified_alignments,N_unqualified,Median_max_local_identity,Median_query_coverage,Full_sequence_exact_match_N,Full_sequence_exact_match_%,Local_identity_ge_0_80_N,Local_identity_ge_0_90_N,Local_identity_ge_0_80_%_of_all,Local_identity_ge_0_90_%_of_all
0,AMP,229,200,29,0.580128,0.823529,0,0.0,28,10,12.227074,4.366812
1,Cyclic,229,227,2,0.900000,0.833333,0,0.0,187,116,81.659389,50.655022
2,MHC,229,222,7,0.428571,0.777778,0,0.0,12,4,5.240175,1.746725


In [10]:
#@title 📊 5. Benchmark Baselines { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-baseline {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-baseline {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-baseline {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-baseline:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-baseline" id="headerBox_baseline">
    <div class="title-baseline">📊 5. Benchmark Baselines — Selected Database Reference &amp; Literature Ranges</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Two reference baselines, computed together here:
        <ul>
        <li><b>Selected database reference distribution</b> per Category × Property (mean, SD, P5/P25/Median/P75/P95, Shapiro-Wilk
        normality check) — used later for statistical comparison.</li>
        <li><b>Literature Compliance Table</b> — fixed, absolute thresholds from published studies, independent of the specific
        Selected reference subset uploaded):</li>
        </ul>
        <table style="border-collapse:collapse;width:100%;font-size:13px;">
        <tr style="background:#0284c7;color:white;"><th style="padding:6px;">Property</th><th style="padding:6px;">Category</th><th style="padding:6px;">Range</th><th style="padding:6px;">Source</th></tr>
        <tr style="background:#e0f7ff;"><td style="padding:6px;text-align:center;">Length</td><td style="padding:6px;text-align:center;">AMP</td><td style="padding:6px;text-align:center;">10&ndash;50 aa</td><td style="padding:6px;text-align:center;">PMC3910807</td></tr>
        <tr><td style="padding:6px;text-align:center;">Length</td><td style="padding:6px;text-align:center;">MHC</td><td style="padding:6px;text-align:center;">typically 9-mer (8&ndash;10 ≈ 78%)</td><td style="padding:6px;text-align:center;">PMC6224037</td></tr>
        <tr style="background:#e0f7ff;"><td style="padding:6px;text-align:center;">Length</td><td style="padding:6px;text-align:center;">Cyclic</td><td style="padding:6px;text-align:center;">7&ndash;40 aa</td><td style="padding:6px;text-align:center;">PLOS Comp Biol 10.1371/journal.pcbi.1012290</td></tr>
        <tr><td style="padding:6px;text-align:center;">Net Charge</td><td style="padding:6px;text-align:center;">AMP</td><td style="padding:6px;text-align:center;">+2 to +9</td><td style="padding:6px;text-align:center;">Springer 10.1186/s40779-021-00343-2</td></tr>
        <tr style="background:#fff7ed;"><td colspan="4" style="padding:6px;">No universal pI compliance interval is applied: 3.38&ndash;10.17 has not been established as a task-specific biological cutoff. pI remains descriptive.</td></tr>
        </table>
        <br>GRAVY, Molecular Weight, and Aliphatic Index have no universal fixed range (task-dependent) — reported as
        descriptive statistics only. For cyclic peptides, MW, pI, and net charge are sequence-derived exploratory values only; primary comparisons use Length, GRAVY, and Aliphatic Index.
    </div>
</div>
<button class="toggle-btn-baseline" onclick="toggleHeader_baseline()">Hide card</button>
<script>
function toggleHeader_baseline() {
    const box = document.getElementById("headerBox_baseline");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


natural_ref_rows = []
for category in sorted(df["Category"].unique()):
    nat_vals_all = df[(df["Category"] == category) & (df["Model"] == "Natural")]
    for prop in PROPERTIES:
        vals = nat_vals_all[prop].dropna().values
        n = len(vals)
        if n == 0:
            continue
        p5, p25, p50, p75, p95 = np.percentile(vals, [5, 25, 50, 75, 95]) if n >= 2 else (np.nan,) * 5
        if n >= 3:
            shapiro_stat, shapiro_p = stats.shapiro(vals)
            is_normal = shapiro_p > 0.05
        else:
            shapiro_stat, shapiro_p, is_normal = np.nan, np.nan, np.nan
        natural_ref_rows.append({
            "Category": category, "Property": prop, "N_Natural": n,
            "Mean": round(np.mean(vals), 3), "SD": round(np.std(vals), 3),
            "P5": round(p5, 3) if n >= 2 else np.nan, "P25": round(p25, 3) if n >= 2 else np.nan,
            "Median": round(p50, 3) if n >= 2 else np.nan, "P75": round(p75, 3) if n >= 2 else np.nan,
            "P95": round(p95, 3) if n >= 2 else np.nan,
            "Shapiro_p": round(shapiro_p, 4) if n >= 3 else np.nan, "Normal_Distribution": is_normal,
        })
natural_ref_df = pd.DataFrame(natural_ref_rows)

# Task-specific fixed thresholds only. The old universal pI 3.38-10.17 interval
# was a calculation-method citation, not justified as biological compliance for all tasks.
LITERATURE_RANGES = {
    "AMP":    {"Length": (10, 50), "NetCharge": (2, 9)},
    "MHC":    {"Length": (8, 10)},
    "Cyclic": {"Length": (7, 40)},
    "Mixed":  {},
}

def literature_compliance_table(df):
    rows = []
    for (category, model), grp in df.groupby(["Category", "Model"]):
        ranges = LITERATURE_RANGES.get(category, {})
        row = {"Category": category, "Model": model, "N": len(grp)}
        for prop in ["Length", "NetCharge", "pI"]:
            row[f"Avg_{prop}"] = round(grp[prop].mean(), 3)
            if prop in ranges:
                lo, hi = ranges[prop]
                row[f"%_{prop}_in_[{lo}-{hi}]"] = round(np.mean((grp[prop] >= lo) & (grp[prop] <= hi)) * 100, 1)
            else:
                row[f"%_{prop}_in_range"] = np.nan
        row["Avg_GRAVY"] = round(grp["GRAVY"].mean(), 3)
        row["Avg_MW"] = round(grp["MW"].mean(), 1)
        row["Avg_AliphaticIndex"] = round(grp["AliphaticIndex"].mean(), 2)
        if category == "MHC":
            row["%_exactly_9mer"] = round(np.mean(grp["Length"] == 9) * 100, 1)
        rows.append(row)
    out = pd.DataFrame(rows)
    out["_natural_first"] = out["Model"] != "Natural"
    return out.sort_values(["Category", "_natural_first", "Model"]).drop(columns="_natural_first")

literature_table = literature_compliance_table(df)
# Per-row warning attached to ALL cyclic descriptive values in exported tables.
# These values are NOT topology-resolved molecular quantities.
literature_table["Cyclic_MW_pI_Charge_Note"] = np.where(
    literature_table["Category"].eq("Cyclic"),
    "MW/pI/NetCharge: exploratory sequence-only; not molecular/topology-aware", ""
)
natural_ref_df["Cyclic_Descriptor_Status"] = np.where(
    natural_ref_df["Category"].eq("Cyclic") & natural_ref_df["Property"].isin(CYCLIC_EXPLORATORY_ONLY),
    "EXPLORATORY sequence-only; not topology-aware", "Primary/standard sequence descriptor"
)

print("Selected database reference distribution:"); display(natural_ref_df)
print("\nLiterature Compliance Table:"); display(literature_table)


Selected database reference distribution:


,Category,Property,N_Natural,Mean,SD,P5,P25,Median,P75,P95,Shapiro_p,Normal_Distribution,Cyclic_Descriptor_Status
0,AMP,Length,229,36.245,25.673,13.000,21.000,31.000,42.000,91.200,0.0000,False,Primary/standard sequence descriptor
1,AMP,MW,229,3998.022,2870.029,1396.492,2333.684,3255.956,4585.304,9909.438,0.0000,False,Primary/standard sequence descriptor
2,AMP,GRAVY,229,-0.003,0.804,-1.061,-0.567,0.008,0.507,1.405,0.0103,False,Primary/standard sequence descriptor
3,AMP,pI,229,9.171,1.754,5.214,8.721,9.390,10.003,11.706,0.0000,False,Primary/standard sequence descriptor
4,AMP,NetCharge,229,3.245,3.412,-1.244,1.709,2.793,4.762,8.808,0.0000,False,Primary/standard sequence descriptor
5,AMP,AliphaticIndex,229,89.728,45.429,23.932,56.316,87.500,118.889,170.970,0.0029,False,Primary/standard sequence descriptor
6,Cyclic,Length,229,8.284,1.888,6.000,6.000,8.000,10.000,11.000,0.0000,False,Primary/standard sequence descriptor
7,Cyclic,MW,229,885.679,175.636,639.611,730.934,877.165,1033.347,1153.463,0.0030,False,EXPLORATORY sequence-only; not topology-aware
8,Cyclic,GRAVY,229,1.801,0.662,0.836,1.470,1.879,2.127,2.737,0.0000,False,Primary/standard sequence descriptor
9,Cyclic,pI,229,5.708,0.531,5.495,5.525,5.525,5.955,5.955,0.0000,False,EXPLORATORY sequence-only; not topology-aware



Literature Compliance Table:


,Category,Model,N,Avg_Length,%_Length_in_[10-50],Avg_NetCharge,%_NetCharge_in_[2-9],Avg_pI,%_pI_in_range,Avg_GRAVY,Avg_MW,Avg_AliphaticIndex,%_Length_in_[7-40],%_NetCharge_in_range,%_Length_in_[8-10],%_exactly_9mer,%_Length_in_range,Cyclic_MW_pI_Charge_Note
5,AMP,Natural,229,36.245,82.1,3.245,59.8,9.171,NaN,-0.003,3998.0,89.73,NaN,NaN,NaN,NaN,NaN,
0,AMP,AMP-RL,6,19.333,100.0,7.774,66.7,11.412,NaN,-1.277,2515.9,71.98,NaN,NaN,NaN,NaN,NaN,
1,AMP,AMPAINTER,30,21.733,100.0,8.024,70.0,11.797,NaN,-0.727,2846.7,94.21,NaN,NaN,NaN,NaN,NaN,
2,AMP,DIFF-AMP,6,14.500,66.7,1.442,50.0,8.047,NaN,0.519,1528.5,85.33,NaN,NaN,NaN,NaN,NaN,
3,AMP,HMAMP,10,19.500,80.0,3.463,80.0,10.525,NaN,0.144,2345.3,117.83,NaN,NaN,NaN,NaN,NaN,
4,AMP,MOQA,4,23.000,100.0,6.276,100.0,10.738,NaN,0.059,2490.8,117.04,NaN,NaN,NaN,NaN,NaN,
6,AMP,PEPGAN,6,19.833,100.0,5.420,100.0,10.867,NaN,0.529,2215.1,145.29,NaN,NaN,NaN,NaN,NaN,
7,AMP,RR-ADS,15,16.333,100.0,5.175,100.0,11.673,NaN,-0.425,2149.6,113.74,NaN,NaN,NaN,NaN,NaN,
10,Cyclic,Natural,229,8.284,NaN,-0.168,NaN,5.708,NaN,1.801,885.7,182.16,69.0,NaN,NaN,NaN,NaN,MW/pI/NetCharge: exploratory sequence-only; no...
8,Cyclic,CYC_BUILDER,9,11.222,NaN,-1.946,NaN,4.373,NaN,-0.698,1337.1,79.47,100.0,NaN,NaN,NaN,NaN,MW/pI/NetCharge: exploratory sequence-only; no...


In [11]:
#@title 📈 6. Model-Level Descriptive Comparison { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-modelcmp {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-modelcmp {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-modelcmp {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-modelcmp:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-modelcmp" id="headerBox_modelcmp">
    <div class="title-modelcmp">📈 6. Model-Level Descriptive Comparison vs. Reference</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        For each Model, computes Mann-Whitney U, Kolmogorov-Smirnov, normalized Wasserstein distance, and empirical
        P5&ndash;P95 compliance against the Selected database reference, then combines these into a composite ranking (Natural itself
        is shown as the 100%-compliance baseline, never ranked as a competitor).
    </div>
</div>
<button class="toggle-btn-modelcmp" onclick="toggleHeader_modelcmp()">Hide card</button>
<script>
function toggleHeader_modelcmp() {
    const box = document.getElementById("headerBox_modelcmp");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


comparison_rows = []
for category in sorted(df["Category"].unique()):
    nat_sub = df[(df["Category"] == category) & (df["Model"] == "Natural")]
    models = [m for m in df[df["Category"] == category]["Model"].unique() if m != "Natural"]
    for prop in primary_properties(category):
        nat_vals = nat_sub[prop].dropna().values
        if len(nat_vals) < 3:
            continue
        p5, p95 = np.percentile(nat_vals, [5, 95])
        nat_std = np.std(nat_vals) if np.std(nat_vals) > 0 else np.nan
        for model in models:
            model_vals = df[(df["Category"] == category) & (df["Model"] == model)][prop].dropna().values
            if len(model_vals) < 3:
                continue
            mw_u, mw_p = stats.mannwhitneyu(model_vals, nat_vals, alternative="two-sided")
            ks_stat, ks_p = stats.ks_2samp(model_vals, nat_vals)
            wass = stats.wasserstein_distance(model_vals, nat_vals)
            wass_norm = wass / nat_std if nat_std and not np.isnan(nat_std) else np.nan
            compliance = float(np.mean((model_vals >= p5) & (model_vals <= p95)) * 100)
            comparison_rows.append({
                "Category": category, "Model": model, "Property": prop,
                "N_Model": len(model_vals), "N_Natural": len(nat_vals),
                "MannWhitney_p": float(mw_p), "Significant_Diff (p<0.05)": mw_p < 0.05,
                "KS_stat": round(ks_stat, 3), "KS_p": float(ks_p),
                "Wasserstein_norm": round(wass_norm, 3) if not np.isnan(wass_norm) else np.nan,
                "Compliance_%_in_Natural_P5-P95": round(compliance, 1),
            })
comparison_df = pd.DataFrame(comparison_rows)

# Multiple-testing families: independently within (Category, Test), across all
# eligible model x primary-descriptor comparisons. BH controls FDR within each family.
def bh_adjust(pvalues):
    vals = np.asarray(pvalues, dtype=float)
    order = np.argsort(vals)
    sorted_p = vals[order]
    q_sorted = np.minimum.accumulate((sorted_p * len(vals) / np.arange(1, len(vals)+1))[::-1])[::-1]
    result = np.empty(len(vals), dtype=float)
    result[order] = np.clip(q_sorted, 0, 1)
    return result

for _category, _idx in comparison_df.groupby("Category").groups.items():
    for _base, _out in [("MannWhitney_p", "MannWhitney_q_BH"), ("KS_p", "KS_q_BH")]:
        comparison_df.loc[_idx, _out] = bh_adjust(comparison_df.loc[_idx, _base].values)
comparison_df["Significant_Diff_BH_FDR_0.05"] = comparison_df["MannWhitney_q_BH"] < 0.05

composite_rows = []
for (category, model), grp in comparison_df.groupby(["Category", "Model"]):
    composite_rows.append({
        "Category": category, "Model": model,
        "Composite_Compliance_%": round(grp["Compliance_%_in_Natural_P5-P95"].mean(), 1),
        "Composite_Distance": round(grp["Wasserstein_norm"].mean(skipna=True), 3),
        "Properties_Tested": len(grp),
        "N_sequences": int(len(df[(df["Category"] == category) & (df["Model"] == model)])),
    })
for category in sorted(df["Category"].unique()):
    composite_rows.append({
        "Category": category, "Model": "Natural", "Composite_Compliance_%": 100.0,
        "Composite_Distance": 0.0, "Properties_Tested": len(primary_properties(category)),
        "N_sequences": int(len(df[(df["Category"] == category) & (df["Model"] == "Natural")])),
    })
composite_df = pd.DataFrame(composite_rows)

# Non-competitive, descriptive model table: deliberately no rank or winner labels.
model_summary_df = composite_df[composite_df["Model"] != "Natural"].copy()
model_summary_df["Low_N_Flag"] = np.where(model_summary_df["N_sequences"] < 10,
                                              "LOW N (<10): exploratory only", "")
model_summary_df = model_summary_df.sort_values(["Category", "Model"])
# Legacy variable kept for plotting compatibility; it has NO ranking columns.
rl_rank_df = model_summary_df.copy()
rank_df = composite_df.copy()
best_per_category = pd.DataFrame(columns=["Category", "Model"])
print("Descriptive model comparisons (no competitive ranks or winners):")
display(model_summary_df)

# Reviewer sensitivity audit: six sequence-derived descriptors, the leave-pI-out five,
# and the three topology-robust composition descriptors, using IDENTICAL input records.
# The six-descriptor result is an intentionally chemically naive counterfactual,
# never a corrected molecular ranking or a primary inferential conclusion.
def cyclic_descriptor_sensitivity(dataset):
    natural = dataset[(dataset["Category"] == "Cyclic") & (dataset["Model"] == "Natural")]
    generated = dataset[(dataset["Category"] == "Cyclic") & (dataset["Model"] != "Natural")]
    audit = []
    regimes = [("6 sequence-derived (exploratory; chemically naive)", PROPERTIES),
               ("5 sequence-derived (pI excluded; exploratory)", [p for p in PROPERTIES if p != "pI"]),
               ("3 composition-only (primary)", CYCLIC_PRIMARY_PROPERTIES)]
    for regime, feature_set in regimes:
        for model, sub in generated.groupby("Model"):
            feature_distances = []
            for prop in feature_set:
                reference_values = natural[prop].dropna().to_numpy()
                model_values = sub[prop].dropna().to_numpy()
                reference_sd = np.std(reference_values)
                if len(reference_values) < 3 or len(model_values) < 3 or not np.isfinite(reference_sd) or reference_sd <= 0:
                    continue
                feature_distances.append(stats.wasserstein_distance(model_values, reference_values) / reference_sd)
            if len(feature_distances) != len(feature_set):
                raise ValueError(f"Sensitivity audit missing properties for {model}: {regime}")
            audit.append({"Regime": regime, "Model": model,
                          "N": len(sub), "Features": len(feature_distances),
                          "Composite_Distance": round(float(np.mean(feature_distances)), 3)})
    result = pd.DataFrame(audit)
    result["Ordering_within_descriptor_set"] = result.groupby("Regime")["Composite_Distance"].rank(method="min", ascending=True).astype(int)
    return result.sort_values(["Regime", "Model"]).reset_index(drop=True)

cyclic_sensitivity_df = cyclic_descriptor_sensitivity(df)
print("Cyclic descriptive ordering sensitivity (NOT a competitive performance rank): six vs five vs three descriptors")
display(cyclic_sensitivity_df)


Descriptive model comparisons (no competitive ranks or winners):


,Category,Model,Composite_Compliance_%,Composite_Distance,Properties_Tested,N_sequences,Low_N_Flag
0,AMP,AMP-RL,75.0,0.987,6,6,LOW N (<10): exploratory only
1,AMP,AMPAINTER,75.6,0.850,6,30,
2,AMP,DIFF-AMP,83.3,0.684,6,6,LOW N (<10): exploratory only
3,AMP,HMAMP,88.3,0.551,6,10,
4,AMP,MOQA,100.0,0.775,6,4,LOW N (<10): exploratory only
5,AMP,PEPGAN,97.2,0.876,6,6,LOW N (<10): exploratory only
6,AMP,RR-ADS,90.0,0.812,6,15,
7,Cyclic,CYC_BUILDER,40.7,2.387,3,9,LOW N (<10): exploratory only
8,Cyclic,HIGHPLAY,8.9,3.318,3,30,


Cyclic descriptive ordering sensitivity (NOT a competitive performance rank): six vs five vs three descriptors


,Regime,Model,N,Features,Composite_Distance,Ordering_within_descriptor_set
0,3 composition-only (primary),CYC_BUILDER,9,3,2.387,1
1,3 composition-only (primary),HIGHPLAY,30,3,3.318,2
2,5 sequence-derived (pI excluded; exploratory),CYC_BUILDER,9,5,3.587,1
3,5 sequence-derived (pI excluded; exploratory),HIGHPLAY,30,5,4.382,2
4,6 sequence-derived (exploratory; chemically na...,CYC_BUILDER,9,6,3.434,1
5,6 sequence-derived (exploratory; chemically na...,HIGHPLAY,30,6,3.994,2


In [12]:
#@title 🧭 7. Generation-Strategy Comparison & Ranking { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-stratcmp {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-stratcmp {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-stratcmp {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-stratcmp:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-stratcmp" id="headerBox_stratcmp">
    <div class="title-stratcmp">🧭 7. Generation-Strategy Comparison &amp; Ranking (NEW)</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Same pipeline as above, but peptides are pooled by <b>Strategy</b> (RL / Pseudo-RL / Hybrid and Advanced /
        Other) instead of individual Model. Descriptive comparison of selected published candidates pooled by strategy;
        pooling does not control for differences in study-specific candidate selection. Kept as a separate
        table/plot from the Model-level results, not merged.
    </div>
</div>
<button class="toggle-btn-stratcmp" onclick="toggleHeader_stratcmp()">Hide card</button>
<script>
function toggleHeader_stratcmp() {
    const box = document.getElementById("headerBox_stratcmp");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


comparison_rows_strat = []
for category in sorted(df["Category"].unique()):
    nat_sub = df[(df["Category"] == category) & (df["Model"] == "Natural")]
    strategies = [s for s in df[df["Category"] == category]["Strategy"].unique() if s != "Natural"]
    for prop in primary_properties(category):
        nat_vals = nat_sub[prop].dropna().values
        if len(nat_vals) < 3:
            continue
        p5, p95 = np.percentile(nat_vals, [5, 95])
        nat_std = np.std(nat_vals) if np.std(nat_vals) > 0 else np.nan
        for strategy in strategies:
            strat_vals = df[(df["Category"] == category) & (df["Strategy"] == strategy)][prop].dropna().values
            if len(strat_vals) < 3:
                continue
            mw_u, mw_p = stats.mannwhitneyu(strat_vals, nat_vals, alternative="two-sided")
            ks_stat, ks_p = stats.ks_2samp(strat_vals, nat_vals)
            wass = stats.wasserstein_distance(strat_vals, nat_vals)
            wass_norm = wass / nat_std if nat_std and not np.isnan(nat_std) else np.nan
            compliance = float(np.mean((strat_vals >= p5) & (strat_vals <= p95)) * 100)
            comparison_rows_strat.append({
                "Category": category, "Strategy": strategy, "Property": prop,
                "N_Strategy": len(strat_vals), "N_Natural": len(nat_vals),
                "MannWhitney_p": float(mw_p), "Significant_Diff (p<0.05)": mw_p < 0.05,
                "KS_stat": round(ks_stat, 3), "KS_p": float(ks_p),
                "Wasserstein_norm": round(wass_norm, 3) if not np.isnan(wass_norm) else np.nan,
                "Compliance_%_in_Natural_P5-P95": round(compliance, 1),
            })
comparison_df_strategy = pd.DataFrame(comparison_rows_strat)

# Same BH method but separate families for strategy-vs-reference tests.
for _category, _idx in comparison_df_strategy.groupby("Category").groups.items():
    for _base, _out in [("MannWhitney_p", "MannWhitney_q_BH"), ("KS_p", "KS_q_BH")]:
        comparison_df_strategy.loc[_idx, _out] = bh_adjust(comparison_df_strategy.loc[_idx, _base].values)
comparison_df_strategy["Significant_Diff_BH_FDR_0.05"] = comparison_df_strategy["MannWhitney_q_BH"] < 0.05

composite_rows_strat = []
for (category, strategy), grp in comparison_df_strategy.groupby(["Category", "Strategy"]):
    composite_rows_strat.append({
        "Category": category, "Strategy": strategy,
        "Composite_Compliance_%": round(grp["Compliance_%_in_Natural_P5-P95"].mean(), 1),
        "Composite_Distance": round(grp["Wasserstein_norm"].mean(skipna=True), 3),
        "Properties_Tested": len(grp),
        "N_sequences": int(len(df[(df["Category"] == category) & (df["Strategy"] == strategy)])),
    })
composite_df_strategy = pd.DataFrame(composite_rows_strat)

# Strategy-level aggregates are descriptive and confounded by study selection.
rank_df_strategy = composite_df_strategy.sort_values(["Category", "Strategy"]).copy()
rank_df_strategy["Low_N_Flag"] = np.where(rank_df_strategy["N_sequences"] < 10,
                                           "LOW N (<10): exploratory only", "")
print("Descriptive strategy summaries (no ranking):")
display(rank_df_strategy)


Descriptive strategy summaries (no ranking):


,Category,Strategy,Composite_Compliance_%,Composite_Distance,Properties_Tested,N_sequences,Low_N_Flag
0,AMP,Hybrid and Advanced,76.8,0.692,6,36,
1,AMP,Pseudo-RL,100.0,0.775,6,4,LOW N (<10): exploratory only
2,AMP,RL,88.3,0.665,6,37,
3,Cyclic,Hybrid and Advanced,16.2,3.101,3,39,


In [13]:
#@title 📏 8. Statistical uncertainty, redundancy and robustness audit
# All results concern SELECTED published candidates; no bootstrap repairs source-selection bias.
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(BOOTSTRAP_SEED)

# Keep intact peptide feature-vectors in each resample: length/MW covariance remains intact.
def composite_from_frames(generated, reference, features, scale="sd", weights=None):
    ds = []
    for prop in features:
        a = generated[prop].dropna().to_numpy(dtype=float)
        b = reference[prop].dropna().to_numpy(dtype=float)
        if len(a) < 3 or len(b) < 3:
            return np.nan
        if scale == "iqr":
            spread = np.percentile(b, 75) - np.percentile(b, 25)
        elif scale == "sd":
            spread = np.std(b)
        else:
            raise ValueError(scale)
        if not np.isfinite(spread) or spread <= 1e-12:
            return np.nan
        ds.append(stats.wasserstein_distance(a, b) / spread)
    return float(np.average(ds, weights=weights)) if ds else np.nan

bootstrap_rows, bootstrap_rank_rows = [], []
for cat in sorted(df["Category"].unique()):
    ref = df[(df.Category == cat) & (df.Model == "Natural")].reset_index(drop=True)
    groups = {model: sub.reset_index(drop=True) for model, sub in
              df[(df.Category == cat) & (df.Model != "Natural")].groupby("Model") if len(sub) >= 3}
    if len(ref) < 3 or not groups:
        continue
    features = primary_properties(cat)
    samples = {model: [] for model in groups}
    for _ in range(BOOTSTRAP_REPS):
        sampled_ref = ref.iloc[rng.integers(0, len(ref), len(ref))]
        for model, sub in groups.items():
            sampled_model = sub.iloc[rng.integers(0, len(sub), len(sub))]
            samples[model].append(composite_from_frames(sampled_model, sampled_ref, features))
    arr = pd.DataFrame(samples)
    # Each column is a model; ranks computed jointly using the same reference draw per replicate.
    rep_ranks = arr.rank(axis=1, ascending=True, method="min")
    for model, sub in groups.items():
        values = arr[model].to_numpy(dtype=float)
        finite = np.isfinite(values)
        obs = composite_from_frames(sub, ref, features)
        rank_values = rep_ranks[model].to_numpy(dtype=float)
        if finite.sum() < BOOTSTRAP_REPS * 0.80:
            ci_lo, ci_hi = np.nan, np.nan
        else:
            ci_lo, ci_hi = np.quantile(values[finite], [.025, .975])
        bootstrap_rows.append({"Category":cat,"Model":model,"N_sequences":len(sub),
            "N_descriptors":len(features),"Composite_Distance_raw":obs,
            "Distance_CI_Lower_95":ci_lo,"Distance_CI_Upper_95":ci_hi,
            "Valid_Bootstrap_reps":int(finite.sum())})
        for ranking in sorted(set(rank_values[np.isfinite(rank_values)])):
            bootstrap_rank_rows.append({"Category":cat,"Model":model,"Bootstrap_Rank":int(ranking),
              "Frequency":float(np.mean(rank_values == ranking))})

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_rank_df = pd.DataFrame(bootstrap_rank_rows)
print("Bootstrap: descriptive uncertainty conditional on selected candidates and reference subset")
display(bootstrap_df.round(4))

# Per-descriptor effect sizes with paired-residue-vector bootstrap uncertainty.
# No CI for n<5: percentile bootstrap with extremely small candidate samples is unstable.
feature_ci_rows = []
feature_rng = np.random.default_rng(BOOTSTRAP_SEED + 109)
for cat in sorted(df["Category"].unique()):
    ref = df[(df.Category == cat) & (df.Model == "Natural")]
    for model, sub in df[(df.Category == cat) & (df.Model != "Natural")].groupby("Model"):
        for prop in primary_properties(cat):
            x = sub[prop].dropna().to_numpy(dtype=float)
            y = ref[prop].dropna().to_numpy(dtype=float)
            n_x, n_y = len(x), len(y)
            std_y = float(np.std(y)) if n_y else np.nan
            effect = (float(stats.wasserstein_distance(x,y)/std_y) if
                      n_x >= 1 and n_y >= 3 and std_y > 1e-12 else np.nan)
            low_n = n_x < 10
            lo = hi = np.nan
            valid = 0
            if n_x >= 5 and n_y >= 5 and np.isfinite(effect):
                replicates = []
                for _ in range(BOOTSTRAP_REPS):
                    xb = x[feature_rng.integers(0,n_x,n_x)]
                    yb = y[feature_rng.integers(0,n_y,n_y)]
                    sd = float(np.std(yb))
                    if sd > 1e-12:
                        replicates.append(float(stats.wasserstein_distance(xb,yb)/sd))
                valid = len(replicates)
                if valid >= BOOTSTRAP_REPS * .80:
                    lo, hi = np.quantile(replicates,[.025,.975])
            feature_ci_rows.append({"Category":cat,"Model":model,"Property":prop,
               "N_sequences":n_x,"N_reference":n_y,"Wasserstein_norm_effect":effect,
               "Effect_CI_Lower_95":lo,"Effect_CI_Upper_95":hi,
               "Bootstrap_reps_valid":valid,"Low_N_Flag": "LOW N (<10)" if low_n else "",
               "CI_Status": "Not estimated (n<5)" if n_x<5 else
                            ("Estimated; interpret cautiously" if np.isfinite(lo) else "Unavailable")})
feature_effect_ci_df = pd.DataFrame(feature_ci_rows)
print("Per-descriptor normalized Wasserstein effect sizes and conditional bootstrap intervals")
display(feature_effect_ci_df.round(4))

# Feature redundancy: reference-only and pooled observations are described separately.
correlation_rows = []
for cat in sorted(df.Category.unique()):
    for population, sub in [("Selected reference",df[(df.Category==cat)&(df.Model=="Natural")]),
                            ("Selected generated",df[(df.Category==cat)&(df.Model!="Natural")])]:
        if len(sub) < 3: continue
        for i, p1 in enumerate(PROPERTIES):
            for p2 in PROPERTIES[i+1:]:
                pair=sub[[p1,p2]].dropna()
                if len(pair)<3 or pair[p1].nunique()<2 or pair[p2].nunique()<2: continue
                rho, pv = stats.spearmanr(pair[p1],pair[p2])
                correlation_rows.append({"Category":cat,"Population":population,"Descriptor_1":p1,
                   "Descriptor_2":p2,"N":len(pair),"Spearman_rho":rho,"p_unadjusted_exploratory":pv})
correlation_df=pd.DataFrame(correlation_rows)
print("Feature redundancy (Spearman; exploratory)")
display(correlation_df[correlation_df.Descriptor_1.eq("Length") & correlation_df.Descriptor_2.eq("MW")].round(4))

# Descriptor/scale/weight sensitivity. No rank here implies unbiased model superiority.
sensitivity_rows=[]
for cat in sorted(df.Category.unique()):
    ref=df[(df.Category==cat)&(df.Model=="Natural")]
    base=primary_properties(cat)
    modes=[("Primary, reference SD, equal weight",base,"sd",None),
           ("Reference IQR, equal weight",base,"iqr",None)]
    if "MW" in base:
        modes.append(("MW excluded, reference SD",[p for p in base if p!="MW"],"sd",None))
        # Combined Length and MW group receives the weight of one descriptor.
        weights=[0.5 if p in ("Length","MW") else 1.0 for p in base]
        modes.append(("Length/MW downweighted as correlated pair",base,"sd",weights))
    for name, props, scale, weights in modes:
        for model,sub in df[(df.Category==cat)&(df.Model!="Natural")].groupby("Model"):
            if len(sub)<3: continue
            score=composite_from_frames(sub,ref,props,scale,weights)
            sensitivity_rows.append({"Category":cat,"Scenario":name,"Model":model,
                   "N_sequences":len(sub),"N_descriptors":len(props),"Composite_Distance":score})
robustness_df=pd.DataFrame(sensitivity_rows)
if not robustness_df.empty:
    robustness_df["Low_N_Flag"] = np.where(robustness_df["N_sequences"] < 10, "LOW N (<10)", "")
print("Normalization, redundancy and weighting sensitivity")
display(robustness_df.round(4))

# Annotate original descriptive ordering (no competitive inferential ranking claimed).
if not bootstrap_df.empty:
    rl_rank_df=rl_rank_df.merge(bootstrap_df[["Category","Model","Distance_CI_Lower_95",
           "Distance_CI_Upper_95","Valid_Bootstrap_reps"]],
           on=["Category","Model"],how="left")
    rank_df=rank_df.merge(bootstrap_df[["Category","Model","Distance_CI_Lower_95",
           "Distance_CI_Upper_95"]],on=["Category","Model"],how="left")
# Report all identified candidates, including n=1/2 models excluded from inferential testing.
model_sample_audit_df = (df[df.Model != "Natural"].groupby(["Category","Model"])
                         .size().reset_index(name="N_sequences"))
model_sample_audit_df["Inference_Status"] = np.where(model_sample_audit_df.N_sequences < 3,
    "DESCRIPTIVE ONLY: no tests or composite score", "Exploratory tests; power not assured")
model_sample_audit_df["Low_N_Flag"] = np.where(model_sample_audit_df.N_sequences < 10,
    "LOW N (<10)", "")
display(model_sample_audit_df)
print("Bootstrap audit complete; selected-candidate uncertainty does not address selection bias.")


Bootstrap: descriptive uncertainty conditional on selected candidates and reference subset


,Category,Model,N_sequences,N_descriptors,Composite_Distance_raw,Distance_CI_Lower_95,Distance_CI_Upper_95,Valid_Bootstrap_reps
0,AMP,AMP-RL,6,6,0.9873,0.8829,1.1994,2000
1,AMP,AMPAINTER,30,6,0.8502,0.7525,1.0532,2000
2,AMP,DIFF-AMP,6,6,0.6846,0.5678,1.1326,2000
3,AMP,HMAMP,10,6,0.5513,0.4828,0.7816,2000
4,AMP,MOQA,4,6,0.7743,0.7449,0.9373,2000
5,AMP,PEPGAN,6,6,0.8762,0.8212,1.0117,2000
6,AMP,RR-ADS,15,6,0.8121,0.7437,0.9235,2000
7,Cyclic,CYC_BUILDER,9,3,2.3870,1.8496,3.0340,2000
8,Cyclic,HIGHPLAY,30,3,3.3178,2.9586,3.7982,2000


Per-descriptor normalized Wasserstein effect sizes and conditional bootstrap intervals


,Category,Model,Property,N_sequences,N_reference,Wasserstein_norm_effect,Effect_CI_Lower_95,Effect_CI_Upper_95,Bootstrap_reps_valid,Low_N_Flag,CI_Status
0,AMP,AMP-RL,Length,6,229,0.6672,0.4789,0.9449,2000,LOW N (<10),Estimated; interpret cautiously
1,AMP,AMP-RL,MW,6,229,0.5484,0.4218,0.8063,2000,LOW N (<10),Estimated; interpret cautiously
2,AMP,AMP-RL,GRAVY,6,229,1.5987,1.0158,2.2599,2000,LOW N (<10),Estimated; interpret cautiously
3,AMP,AMP-RL,pI,6,229,1.2861,1.0568,1.5746,2000,LOW N (<10),Estimated; interpret cautiously
4,AMP,AMP-RL,NetCharge,6,229,1.3459,0.8607,2.2596,2000,LOW N (<10),Estimated; interpret cautiously
...,...,...,...,...,...,...,...,...,...,...,...
79,Mixed,VAE-MH,MW,15,0,NaN,NaN,NaN,0,,Unavailable
80,Mixed,VAE-MH,GRAVY,15,0,NaN,NaN,NaN,0,,Unavailable
81,Mixed,VAE-MH,pI,15,0,NaN,NaN,NaN,0,,Unavailable
82,Mixed,VAE-MH,NetCharge,15,0,NaN,NaN,NaN,0,,Unavailable


Feature redundancy (Spearman; exploratory)


,Category,Population,Descriptor_1,Descriptor_2,N,Spearman_rho,p_unadjusted_exploratory
0,AMP,Selected reference,Length,MW,229,0.9912,0.0
15,AMP,Selected generated,Length,MW,77,0.8817,0.0
30,Cyclic,Selected reference,Length,MW,229,0.9434,0.0
45,Cyclic,Selected generated,Length,MW,39,0.9543,0.0
60,MHC,Selected reference,Length,MW,229,0.4159,0.0
85,Mixed,Selected generated,Length,MW,29,0.9907,0.0


Normalization, redundancy and weighting sensitivity


,Category,Scenario,Model,N_sequences,N_descriptors,Composite_Distance,Low_N_Flag
0,AMP,"Primary, reference SD, equal weight",AMP-RL,6,6,0.9873,LOW N (<10)
1,AMP,"Primary, reference SD, equal weight",AMPAINTER,30,6,0.8502,
2,AMP,"Primary, reference SD, equal weight",DIFF-AMP,6,6,0.6846,LOW N (<10)
3,AMP,"Primary, reference SD, equal weight",HMAMP,10,6,0.5513,
4,AMP,"Primary, reference SD, equal weight",MOQA,4,6,0.7743,LOW N (<10)
5,AMP,"Primary, reference SD, equal weight",PEPGAN,6,6,0.8762,LOW N (<10)
6,AMP,"Primary, reference SD, equal weight",RR-ADS,15,6,0.8121,
7,AMP,"Reference IQR, equal weight",AMP-RL,6,6,1.0538,LOW N (<10)
8,AMP,"Reference IQR, equal weight",AMPAINTER,30,6,0.9689,
9,AMP,"Reference IQR, equal weight",DIFF-AMP,6,6,0.7738,LOW N (<10)


,Category,Model,N_sequences,Inference_Status,Low_N_Flag
0,AMP,AMP-RL,6,Exploratory tests; power not assured,LOW N (<10)
1,AMP,AMPAINTER,30,Exploratory tests; power not assured,
2,AMP,DIFF-AMP,6,Exploratory tests; power not assured,LOW N (<10)
3,AMP,HMAMP,10,Exploratory tests; power not assured,
4,AMP,MOQA,4,Exploratory tests; power not assured,LOW N (<10)
5,AMP,PEPGAN,6,Exploratory tests; power not assured,LOW N (<10)
6,AMP,RR-ADS,15,Exploratory tests; power not assured,
7,Cyclic,CYC_BUILDER,9,Exploratory tests; power not assured,LOW N (<10)
8,Cyclic,HIGHPLAY,30,Exploratory tests; power not assured,
9,MHC,RLPMIEC,2,DESCRIPTIVE ONLY: no tests or composite score,LOW N (<10)


Bootstrap audit complete; selected-candidate uncertainty does not address selection bias.


In [14]:
#@title 🧪 8A. Near-similarity cluster sensitivity (0.80 and 0.70)
# Run after model comparison and bootstrap cells; source data and main inference are NEVER altered.
# Single-linkage connected components within each generated (Category, Model) only.
# Deterministically retain the earliest parsed row per component. This is an illustrative
# reduction of within-model sequence redundancy, NOT correction for unknown common runs.
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

assert "near_similarity_all_df" in globals(), "Run Cell 4A first."
assert "comparison_df" in globals() and "bh_adjust" in globals(), "Run model-comparison cell first."

def cluster_representatives(dataset, threshold):
    keep_indices = []
    cluster_rows = []
    for (category, model), group in dataset[dataset.Model != "Natural"].groupby(["Category","Model"], sort=True):
        inds = list(group.index)
        parent = {idx:idx for idx in inds}
        def root(v):
            while parent[v] != v:
                parent[v] = parent[parent[v]]
                v = parent[v]
            return v
        def union(i,j):
            ri,rj=root(i),root(j)
            if ri != rj: parent[max(ri,rj)] = min(ri,rj)
        # Use integer row positions via the stable record IDs, allowing repeated IDs safely.
        sequences = [(idx, str(dataset.at[idx,"CleanSeq"]).upper()) for idx in inds]
        for (ia,sa),(ib,sb) in combinations(sequences,2):
            sim=1-_levenshtein_distance(sa,sb)/max(len(sa),len(sb))
            if sim + 1e-12 >= threshold: union(ia,ib)
        components={}
        for idx in inds: components.setdefault(root(idx),[]).append(idx)
        for members in components.values():
            rep=min(members)
            keep_indices.append(rep)
            cluster_rows.append({"Threshold":threshold,"Category":category,"Model":model,
                                 "Cluster_representative_ID":dataset.at[rep,"ID"],
                                 "Cluster_size":len(members),
                                 "Member_IDs":" | ".join(str(dataset.at[k,"ID"]) for k in members)})
    # all reference rows retained; only within-model generated sequence redundancy reduced
    keep_indices.extend(dataset.index[dataset.Model == "Natural"].tolist())
    return dataset.loc[sorted(keep_indices)].copy(), pd.DataFrame(cluster_rows)

sensitivity_comparison_rows=[]
cluster_audit_rows=[]
cluster_members_tables=[]
for threshold in [0.80,0.70]:
    reduced, member_table = cluster_representatives(df,threshold)
    cluster_members_tables.append(member_table)
    for (category,model),original in df[df.Model != "Natural"].groupby(["Category","Model"]):
        n_reduced=int(((reduced.Category==category)&(reduced.Model==model)).sum())
        cluster_audit_rows.append({"Threshold":threshold,"Category":category,"Model":model,
                                   "N_original":len(original),"N_one_per_cluster":n_reduced,
                                   "Sequences_removed_for_sensitivity_only":len(original)-n_reduced,
                                   "Has_sufficient_n_after_reduction":n_reduced>=3})
    # Recalculate *all* category x model x descriptor tests before applying BH,
    # because altering one model changes the number of tests in the family.
    calc=[]
    for category in sorted(df.Category.unique()):
        ref = reduced[(reduced.Category==category)&(reduced.Model=="Natural")]
        if len(ref)<3:continue
        for model,sub in reduced[(reduced.Category==category)&(reduced.Model!="Natural")].groupby("Model"):
            if len(sub)<3:continue
            for prop in primary_properties(category):
                a=sub[prop].dropna().to_numpy(dtype=float)
                b=ref[prop].dropna().to_numpy(dtype=float)
                if len(a)<3 or len(b)<3:continue
                scale=np.std(b)
                calc.append({"Threshold":threshold,"Category":category,"Model":model,"Property":prop,
                             "N_after":len(a),"Wasserstein_norm_after":stats.wasserstein_distance(a,b)/scale if scale>0 else np.nan,
                             "MannWhitney_p_after":stats.mannwhitneyu(a,b,alternative="two-sided").pvalue,
                             "KS_p_after":stats.ks_2samp(a,b).pvalue})
    adjusted=pd.DataFrame(calc)
    if not adjusted.empty:
        for cat,idx in adjusted.groupby("Category").groups.items():
            for pcol,qcol in [("MannWhitney_p_after","MannWhitney_q_BH_after"),("KS_p_after","KS_q_BH_after")]:
                adjusted.loc[idx,qcol]=bh_adjust(adjusted.loc[idx,pcol].to_numpy(dtype=float))
        orig=comparison_df[["Category","Model","Property","Wasserstein_norm","MannWhitney_q_BH","KS_q_BH"]]
        adjusted=adjusted.merge(orig,on=["Category","Model","Property"],how="left")
        adjusted["Distance_change_after_minus_before"]=adjusted.Wasserstein_norm_after-adjusted.Wasserstein_norm
        sensitivity_comparison_rows.extend(adjusted.to_dict("records"))
cluster_sensitivity_counts_df=pd.DataFrame(cluster_audit_rows)
cluster_sensitivity_comparisons_df=pd.DataFrame(sensitivity_comparison_rows)
cluster_members_df=pd.concat(cluster_members_tables,ignore_index=True)
# Compare composite averages over the same descriptor set before/after reduction.
if not cluster_sensitivity_comparisons_df.empty:
    composite_cluster_sensitivity_df=(cluster_sensitivity_comparisons_df.groupby(["Threshold","Category","Model"],as_index=False)
        .agg(Composite_before=("Wasserstein_norm","mean"),Composite_after=("Wasserstein_norm_after","mean"),
             N_after=("N_after","first"),N_descriptors=("Property","nunique")))
    composite_cluster_sensitivity_df["Delta_after_minus_before"]=composite_cluster_sensitivity_df.Composite_after-composite_cluster_sensitivity_df.Composite_before
else:
    composite_cluster_sensitivity_df=pd.DataFrame(columns=["Threshold","Category","Model","Composite_before","Composite_after","N_after","N_descriptors","Delta_after_minus_before"])
for filename,table in [("near_similarity_cluster_counts",cluster_sensitivity_counts_df),
                       ("near_similarity_cluster_members",cluster_members_df),
                       ("near_similarity_cluster_descriptor_q_comparison",cluster_sensitivity_comparisons_df),
                       ("near_similarity_cluster_composite_comparison",composite_cluster_sensitivity_df)]:
    table.to_csv(filename+".csv",index=False)
print("Exploratory threshold sensitivity; all source records retained in main analysis.")
display(cluster_sensitivity_counts_df)
display(composite_cluster_sensitivity_df.round(4))
print("Thresholds are heuristics; no sequence-based method establishes shared-run independence.")


Exploratory threshold sensitivity; all source records retained in main analysis.


,Threshold,Category,Model,N_original,N_one_per_cluster,Sequences_removed_for_sensitivity_only,Has_sufficient_n_after_reduction
0,0.8,AMP,AMP-RL,6,6,0,True
1,0.8,AMP,AMPAINTER,30,29,1,True
2,0.8,AMP,DIFF-AMP,6,6,0,True
3,0.8,AMP,HMAMP,10,10,0,True
4,0.8,AMP,MOQA,4,4,0,True
5,0.8,AMP,PEPGAN,6,6,0,True
6,0.8,AMP,RR-ADS,15,15,0,True
7,0.8,Cyclic,CYC_BUILDER,9,7,2,True
8,0.8,Cyclic,HIGHPLAY,30,16,14,True
9,0.8,MHC,RLPMIEC,2,2,0,False


,Threshold,Category,Model,Composite_before,Composite_after,N_after,N_descriptors,Delta_after_minus_before
0,0.7,AMP,AMP-RL,0.9873,0.9873,6,6,0.0000
1,0.7,AMP,AMPAINTER,0.8502,0.8409,28,6,-0.0093
2,0.7,AMP,DIFF-AMP,0.6845,0.6846,6,6,0.0001
3,0.7,AMP,HMAMP,0.5512,0.5513,10,6,0.0001
4,0.7,AMP,MOQA,0.7745,0.7743,4,6,-0.0002
5,0.7,AMP,PEPGAN,0.8762,0.8762,6,6,0.0000
6,0.7,AMP,RR-ADS,0.8122,0.8121,15,6,-0.0000
7,0.7,Cyclic,CYC_BUILDER,2.3873,2.9984,5,3,0.6111
8,0.7,Cyclic,HIGHPLAY,3.3180,3.3501,13,3,0.0321
9,0.8,AMP,AMP-RL,0.9873,0.9873,6,6,0.0000


Thresholds are heuristics; no sequence-based method establishes shared-run independence.


In [15]:
#@title 🧪 8B. Full six-descriptor leave-one-feature-out and extended-pI boundary audit
# This is POST-HOC score sensitivity only; no generative-model retraining occurs.
# For Cyclic, all 6-feature variants are exploratory; the official primary is 3 features.
import numpy as np
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis

lofo_rows = []
for category in sorted(df.Category.unique()):
    reference = df[(df.Category == category) & (df.Model == "Natural")]
    if len(reference) < 3:
        continue
    candidates = df[(df.Category == category) & (df.Model != "Natural")]
    for model, generated in candidates.groupby("Model"):
        if len(generated) < 3:
            continue
        for omitted in [None] + PROPERTIES:
            features = [p for p in PROPERTIES if p != omitted]
            score = composite_from_frames(generated, reference, features, scale="sd")
            lofo_rows.append({"Category": category, "Model": model,
                "Scenario": "All six (baseline)" if omitted is None else "Without " + omitted,
                "Omitted_descriptor": "None" if omitted is None else omitted,
                "N_sequences": len(generated), "N_descriptors": len(features),
                "Composite_Distance": score,
                "Applicability": "Exploratory sequence-only; NOT cyclic molecular properties"
                   if category == "Cyclic" else "Sequence-derived"})
full_lofo_df = pd.DataFrame(lofo_rows)
if not full_lofo_df.empty:
    full_lofo_df["Ordering_within_scenario"] = (
        full_lofo_df.groupby(["Category", "Scenario"])["Composite_Distance"]
        .rank(ascending=True, method="min"))
    full_lofo_df.to_csv("REVISED_full_leave_one_feature_out.csv", index=False)
    print("Full 6+6 leave-one-feature-out cases:", len(full_lofo_df))
    display(full_lofo_df.round(4))

pi_rows = []
for _, rec in df.iterrows():
    analysis = ProteinAnalysis(rec["CleanSeq"])
    default_pi = float(analysis.isoelectric_point())
    extended_pi = float(rec["pI"])
    pi_rows.append({"Category": rec["Category"], "Model": rec["Model"],
        "ID": rec["ID"], "pI_default_Biopython": default_pi,
        "pI_extended_0_14": extended_pi, "Absolute_difference": abs(extended_pi-default_pi),
        "Default_near_4_05": abs(default_pi-4.050028) < 1e-3,
        "Default_near_12": abs(default_pi-12.0) < 1e-3})
pi_comparison_df = pd.DataFrame(pi_rows)
pi_comparison_df.to_csv("REVISED_all_peptide_pI.csv", index=False)
pi_boundary_audit_df = (pi_comparison_df.groupby(["Category", "Model"], dropna=False)
    .agg(N=("ID", "size"), Near_4_05=("Default_near_4_05", "sum"),
         Near_12=("Default_near_12", "sum"),
         Changed_over_0_01=("Absolute_difference", lambda v: int((v > 0.01).sum())))
    .reset_index())
pi_boundary_audit_df.to_csv("REVISED_pI_boundary_audit.csv", index=False)
assert len(pi_comparison_df) == len(df)
assert df["pI"].notna().all(), "pI NaN detected: no silent omissions are allowed"
print("Extended pI audit: records", len(df), "changed >0.01:",
      int((pi_comparison_df.Absolute_difference > 0.01).sum()))
display(pi_boundary_audit_df)


Full 6+6 leave-one-feature-out cases: 63


,Category,Model,Scenario,Omitted_descriptor,N_sequences,N_descriptors,Composite_Distance,Applicability,Ordering_within_scenario
0,AMP,AMP-RL,All six (baseline),None,6,6,0.9873,Sequence-derived,7.0
1,AMP,AMP-RL,Without Length,Length,6,5,1.0514,Sequence-derived,7.0
2,AMP,AMP-RL,Without MW,MW,6,5,1.0751,Sequence-derived,7.0
3,AMP,AMP-RL,Without GRAVY,GRAVY,6,5,0.8651,Sequence-derived,6.0
4,AMP,AMP-RL,Without pI,pI,6,5,0.9276,Sequence-derived,7.0
...,...,...,...,...,...,...,...,...,...
58,Cyclic,HIGHPLAY,Without MW,MW,30,5,3.6651,Exploratory sequence-only; NOT cyclic molecula...,2.0
59,Cyclic,HIGHPLAY,Without GRAVY,GRAVY,30,5,4.1010,Exploratory sequence-only; NOT cyclic molecula...,2.0
60,Cyclic,HIGHPLAY,Without pI,pI,30,5,4.3820,Exploratory sequence-only; NOT cyclic molecula...,2.0
61,Cyclic,HIGHPLAY,Without NetCharge,NetCharge,30,5,3.5300,Exploratory sequence-only; NOT cyclic molecula...,2.0


Extended pI audit: records 836 changed >0.01: 92


,Category,Model,N,Near_4_05,Near_12,Changed_over_0_01
0,AMP,AMP-RL,6,0,1,1
1,AMP,AMPAINTER,30,0,16,16
2,AMP,DIFF-AMP,6,1,0,1
3,AMP,HMAMP,10,0,2,1
4,AMP,MOQA,4,0,0,0
5,AMP,Natural,229,3,7,8
6,AMP,PEPGAN,6,0,1,1
7,AMP,RR-ADS,15,0,7,7
8,Cyclic,CYC_BUILDER,9,4,0,4
9,Cyclic,HIGHPLAY,30,4,0,4


In [16]:
#@title 🎨 8. Visualizations { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-viz {
        background: linear-gradient(135deg, #ffd1dc, #ffc2d1, #ffe4ec);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(255, 182, 193, 0.4);
        margin-bottom: 16px;
    }
    .title-viz {
        text-align: center;
        color: #db2777;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-viz {
        background: white;
        color: #db2777;
        border: 1px solid #fbcfe8;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-viz:hover {
        background: #fce7f3;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-viz" id="headerBox_viz">
    <div class="title-viz">🎨 8. Visualizations</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #db2777, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        One cell defines and immediately renders all five plots: (1) KDE distribution overlay, (2) empirical CDF,
        (3) Mann-Whitney significance heatmap, (4) alphabetical model coverage bars, (5) descriptive strategy bars (NEW). Function
        definitions and their execution are merged since they are only ever used once, right here.
    </div>
</div>
<button class="toggle-btn-viz" onclick="toggleHeader_viz()">Hide card</button>
<script>
function toggleHeader_viz() {
    const box = document.getElementById("headerBox_viz");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return f'<img src="data:image/png;base64,{encoded}" style="max-width:100%;margin:10px 0;">'

def model_order(models):
    return sorted(models, key=lambda m: (m != "Natural", m))

def strategy_order(strategies):
    priority = {"RL": 0, "Pseudo-RL": 1, "Hybrid and Advanced": 2, "Other": 3}
    return sorted(strategies, key=lambda s: priority.get(s, 99))

def plot_distributions(df):
    categories = sorted(df["Category"].unique())
    n_cats, n_props = len(categories), len(PROPERTIES)
    fig, axes = plt.subplots(n_props, n_cats, figsize=(4.5 * n_cats, 3.2 * n_props), squeeze=False)
    fig.suptitle("Distribution Comparison: Selected Database Reference vs Generated Models", fontsize=14, fontweight="bold", y=1.01)
    for col_i, cat in enumerate(categories):
        sub = df[df["Category"] == cat]
        models = model_order(sub["Model"].unique())
        non_natural = [m for m in models if m != "Natural"]
        colors = plt.cm.tab10(np.linspace(0, 0.9, max(len(non_natural), 1)))
        color_map = {m: colors[i] for i, m in enumerate(non_natural)}
        for row_i, prop in enumerate(PROPERTIES):
            ax = axes[row_i][col_i]
            for m in models:
                vals = sub[sub["Model"] == m][prop].dropna().values
                if m != "Natural" and len(vals) < 3:
                    continue
                # Small N: show observed values as rug marks, not a deceptive smooth KDE.
                if m != "Natural" and len(vals) < 10:
                    ax.plot(vals, np.zeros(len(vals)), '|', color=color_map[m],
                            markersize=7, alpha=0.75, label=f"{m} (n={len(vals)}, rug)")
                    continue
                if np.std(vals) == 0:
                    ax.plot(vals, np.zeros(len(vals)), '|', color="#7f8c8d" if m=="Natural" else color_map[m],markersize=7)
                    continue
                try:
                    if m == "Natural":
                        sns.kdeplot(vals, ax=ax, fill=True, color="#7f8c8d", alpha=0.3, linewidth=2, label="Selected database reference")
                    else:
                        sns.kdeplot(vals, ax=ax, fill=False, color=color_map[m], linewidth=1.8, label=m)
                except Exception:
                    continue
            status = " [EXPLORATORY: sequence-only]" if cat == "Cyclic" and prop in CYCLIC_EXPLORATORY_ONLY else ""
            ax.set_title(f"{cat} - {prop}{status}", fontsize=9, fontweight="bold")
            ax.set_xlabel(prop, fontsize=8); ax.set_ylabel("Density", fontsize=8)
            if row_i == 0 and col_i == n_cats - 1:
                ax.legend(fontsize=7, loc="upper right")
            ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig

def plot_ecdf(df):
    categories = sorted(df["Category"].unique())
    n_cats, n_props = len(categories), len(PROPERTIES)
    fig, axes = plt.subplots(n_props, n_cats, figsize=(4.5 * n_cats, 3.2 * n_props), squeeze=False)
    fig.suptitle("Empirical CDF: Selected Database Reference vs Generated Models", fontsize=14, fontweight="bold", y=1.01)
    for col_i, cat in enumerate(categories):
        sub = df[df["Category"] == cat]
        models = model_order(sub["Model"].unique())
        non_natural = [m for m in models if m != "Natural"]
        colors = plt.cm.tab10(np.linspace(0, 0.9, max(len(non_natural), 1)))
        color_map = {m: colors[i] for i, m in enumerate(non_natural)}
        for row_i, prop in enumerate(PROPERTIES):
            ax = axes[row_i][col_i]
            for m in models:
                vals = np.sort(sub[sub["Model"] == m][prop].dropna().values)
                if m != "Natural" and len(vals) < 3:
                    continue
                y = np.arange(1, len(vals) + 1) / len(vals)
                if m == "Natural":
                    ax.step(vals, y, where="post", color="#2c3e50", linewidth=2.5, linestyle="--", label="Selected database reference")
                else:
                    ax.step(vals, y, where="post", color=color_map[m], linewidth=1.6, label=m)
            status = " [EXPLORATORY: sequence-only]" if cat == "Cyclic" and prop in CYCLIC_EXPLORATORY_ONLY else ""
            ax.set_title(f"{cat} - {prop}{status}", fontsize=9, fontweight="bold")
            ax.set_xlabel(prop, fontsize=8); ax.set_ylabel("Cumulative Probability", fontsize=8)
            if row_i == 0 and col_i == n_cats - 1:
                ax.legend(fontsize=7, loc="lower right")
            ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig

def plot_significance_heatmap(comparison_df):
    categories = sorted(comparison_df["Category"].unique())
    # ارتفاع فیگور را بر اساس بیشترین تعداد مدل در هر Category، به‌صورت پویا تنظیم می‌کنیم
    # تا وقتی تعداد مدل‌ها (ردیف‌های heatmap) زیاد است، لیبل‌ها روی هم نیفتند.
    n_models_max = comparison_df.groupby("Category")["Model"].nunique().max()
    fig_height = max(4.5, 0.75 * n_models_max + 1.5)
    fig, axes = plt.subplots(1, len(categories), figsize=(7 * len(categories), fig_height), squeeze=False)
    fig.suptitle("Mann-Whitney U: BH-adjusted q-values vs Selected Reference (by category)", fontsize=13, fontweight="bold")
    for i, cat in enumerate(categories):
        ax = axes[0][i]
        sub = comparison_df[comparison_df["Category"] == cat]
        # Only the category's primary tested properties appear in this inferential heatmap.
        # In particular cyclic MW, pI and charge are not tested or shown.
        pivot = sub.pivot_table(index="Model", columns="Property", values="MannWhitney_q_BH").reindex(columns=primary_properties(cat))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=0.2, linewidths=0.5, ax=ax,
                    cbar_kws={"label": "BH-FDR q-value"})
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_title(f"{cat}\n(q>=0.05 means insufficient evidence of difference, NOT equivalence)", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")
        # --- رفع مشکل هم‌پوشانی لیبل‌های محور Y (اسم مدل‌ها) ---
        ax.tick_params(axis='y', rotation=0, labelsize=10)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        for label in ax.get_xticklabels():
            label.set_ha("right")
    plt.tight_layout()
    return fig

def plot_ranking(rl_rank_df):
    categories = sorted(rl_rank_df["Category"].unique())
    fig, axes = plt.subplots(1, len(categories), figsize=(5.5 * len(categories), 5), squeeze=False)
    fig.suptitle("Descriptive reference-percentile coverage (not performance ranking)", fontsize=13, fontweight="bold")
    for i, cat in enumerate(categories):
        ax = axes[0][i]
        sub = rl_rank_df[rl_rank_df["Category"] == cat].sort_values("Model")
        bars = ax.barh(sub["Model"], sub["Composite_Compliance_%"], color="#2980b9", alpha=0.85)
        for bar, v in zip(bars, sub["Composite_Compliance_%"]):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{v:.1f}%", va="center", fontsize=8)
        ax.set_xlim(0, 110)
        ax.axvline(100, color="#7f8c8d", linestyle="--", linewidth=2, alpha=0.8)
        ax.text(100, len(sub) - 0.3, "Selected database reference (100%)", color="#555555", fontsize=7, ha="right", va="bottom")
        ax.set_xlabel("% of values inside Reference P5-P95 range (avg across properties)", fontsize=8)
        ax.set_title(cat, fontsize=11, fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    return fig

def plot_strategy_ranking(rank_df_strategy):
    categories = sorted(rank_df_strategy["Category"].unique())
    fig, axes = plt.subplots(1, len(categories), figsize=(5.5 * len(categories), 4.5), squeeze=False)
    fig.suptitle("Descriptive strategy reference-percentile coverage (study-confounded)", fontsize=13, fontweight="bold")
    strat_colors = {"RL": "#2980b9", "Pseudo-RL": "#8e44ad", "Hybrid and Advanced": "#16a085", "Other": "#95a5a6"}
    for i, cat in enumerate(categories):
        ax = axes[0][i]
        sub = rank_df_strategy[rank_df_strategy["Category"] == cat].sort_values("Strategy")
        colors = [strat_colors.get(s, "#7f8c8d") for s in sub["Strategy"]]
        bars = ax.barh(sub["Strategy"], sub["Composite_Compliance_%"], color=colors, alpha=0.9)
        for bar, v in zip(bars, sub["Composite_Compliance_%"]):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{v:.1f}%", va="center", fontsize=9)
        ax.set_xlim(0, 110)
        ax.axvline(100, color="#7f8c8d", linestyle="--", linewidth=2, alpha=0.8)
        ax.set_xlabel("% of values inside Reference P5-P95 range (avg across properties)", fontsize=8)
        ax.set_title(cat, fontsize=11, fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    return fig

print("Generating plots...")
img_dist     = fig_to_base64(plot_distributions(df))
img_ecdf     = fig_to_base64(plot_ecdf(df))
img_sig      = fig_to_base64(plot_significance_heatmap(comparison_df))
img_rank     = fig_to_base64(plot_ranking(rl_rank_df))
img_strategy = fig_to_base64(plot_strategy_ranking(rank_df_strategy))
print("Plots done")

Generating plots...
Plots done


In [17]:
#@title 📑 9. Build & Download HTML Report { display-mode: "form" }
from IPython.display import HTML, display
display(HTML(r"""
<style>
    .main-header-report {
        background: linear-gradient(135deg, #bfe6fb, #cdeffd, #e0f7ff);
        border-radius: 22px;
        padding: 22px 26px;
        font-family: 'Segoe UI', Arial, sans-serif;
        color: #1f2937;
        box-shadow: 0 8px 20px rgba(125, 211, 252, 0.4);
        margin-bottom: 16px;
    }
    .title-report {
        text-align: center;
        color: #0284c7;
        margin: 0 0 6px 0;
        font-size: 20px;
        font-weight: bold;
    }
    .toggle-btn-report {
        background: white;
        color: #0284c7;
        border: 1px solid #bae6fd;
        padding: 7px 14px;
        border-radius: 9999px;
        cursor: pointer;
        font-weight: 600;
        font-size: 13px;
        margin: 4px 0 0 0;
        transition: all 0.2s;
    }
    .toggle-btn-report:hover {
        background: #e0f2fe;
        transform: translateY(-1px);
    }
</style>
<div class="main-header-report" id="headerBox_report">
    <div class="title-report">📑 9. Build &amp; Download HTML Report</div>

    <hr style="border:none; height:1px; background:linear-gradient(to right, transparent, #0284c7, transparent); margin:14px 0;">
    <div style="font-size:14px; line-height:1.7; color:#374151;">
        Assembles everything above (Literature Compliance Table, Selected database reference table, Model-level plots/tables, Strategy-level
        plots/tables, full peptide list, method notes) into one downloadable HTML report.
    </div>
</div>
<button class="toggle-btn-report" onclick="toggleHeader_report()">Hide card</button>
<script>
function toggleHeader_report() {
    const box = document.getElementById("headerBox_report");
    if (box.style.display === "none") {
        box.style.display = "block";
    } else {
        box.style.display = "none";
    }
}
</script>
"""))


for _var in ("img_dist", "img_ecdf", "img_sig", "img_rank", "img_strategy"):
    if _var not in globals():
        raise RuntimeError(
            "Run the '9. Visualizations' cell above first (it must finish and print "
            "'Plots done') before running this report cell."
        )

assert "input_integrity_df" in globals(), "Run FASTA audit cell 4A before generating the HTML report."
assert "cluster_sensitivity_counts_df" in globals(), "Run cell 8A before generating the HTML report."
html = """
<html>
<head>
<meta charset="UTF-8">
<style>
body { font-family: Arial, sans-serif; background:linear-gradient(135deg,#fdf2f8 0%,#e0f2fe 100%); margin:20px; color:#334155; }
h1 { color:#db2777; border-bottom:3px solid #0ea5e9; padding-bottom:10px; }
h2 { color:#0369a1; border-bottom:2px solid #fbcfe8; padding-bottom:5px; margin-top:40px; }
h3 { color:#be185d; margin-top:25px; }
p.note { font-size:13px; color:#64748b; }
table {
  border-collapse:collapse; width:100%; margin-bottom:30px;
  background:white; box-shadow:0 2px 8px rgba(219,39,119,0.12); font-size:13px;
}
th { background-color:#0284c7; color:white; padding:10px 8px; white-space:nowrap; }
td { padding:8px; text-align:center; }
tr:nth-child(even) { background:#fdf2f8; }
tr:hover { background:#fce7f3; }
</style>
</head>
<body>
<h1>Property-Based Benchmarking: Generated Peptides vs. Selected Database Reference Peptides</h1>
<p class="note">Two independent descriptions are reported: (1) compliance with the selected task-specific literature-defined ranges (no general pI cutoff), and (2) statistical similarity to the selected database reference set. For Cyclic, the primary statistical comparisons and descriptive composite distances use only Length, GRAVY, and AliphaticIndex because molecule-specific topology and modifications are absent from reference FASTA. MW, pI, and NetCharge in the descriptive tables and plots are sequence-derived exploratory approximations, not chemical properties of reconstructed macrocycles. Results are shown separately by Model and Generation Strategy.</p>
"""

html += "<h2>Extended pI Numerical Audit (Biopython charge model; 0-14)</h2>"
html += "<p class='note'>Expanded pI numerical root search, without default 4.05--12 truncation; modeled as linear peptide sequences. For cyclic sequences this is an exploratory approximation, not a molecular pI.</p>"
html += pi_boundary_audit_df.to_html(index=False)
html += "<h2>Full 6 + 6 Leave-One-Feature-Out Descriptive Sensitivity</h2>"
html += "<p class='note'>For each model with a suitable category-specific reference and n>=3: baseline six-descriptor composite, then six five-descriptor composites, one descriptor omitted each time. No model retraining; cyclic six/five-feature composites are exploratory ONLY. Cyclic primary remains Length, GRAVY and AliphaticIndex.</p>"
html += full_lofo_df.round(4).to_html(index=False)
html += "<h2>Literature Compliance Table</h2>"
html += "<p class='note'>Selected task-specific fixed thresholds only; pI is descriptive and not used for fixed-interval compliance.</p>"
html += literature_table.to_html(index=False)

html += "<p class='note'>For all Cyclic rows below, average MW, pI and NetCharge are EXPLORATORY SEQUENCE-ONLY values and must not be interpreted as molecularly correct cyclic descriptors.</p>"
html += "<h2>Selected Database Reference Distribution (per Category x Property)</h2>"
html += natural_ref_df.to_html(index=False)

html += "<h2>Cyclic Descriptive Ordering under Each Descriptor Set (6 vs 5 without pI vs 3; identical records)</h2>"
html += "<p class='note'><b>Important:</b> Both six- and five-descriptor scenarios are chemically naive SEQUENCE-ONLY sensitivity analyses; removing only pI directly tests the reviewer-described perturbation. Their pI, charge and nominal MW do not describe true cyclic molecular topology. Only the three-composition-descriptor scenario is used for primary cyclic distances. Inspect changes in both distances and ranks under matched records; these tests do not establish any topology-corrected molecular ranking.</p>"
html += cyclic_sensitivity_df.to_html(index=False)
html += "<p class='note'><b>Structure applicability:</b> CYC_BUILDER FHT examples are head-to-tail amide cycles; HIGHPLAY sequences are designed around disulfide cyclization. Disulfide closure does not automatically suppress free termini. Actual terminal caps, side-chain crosslinks, and reference-molecule stereochemistry/modifications cannot be recovered reliably from the simplified cyclic FASTA. Consequently no topology-aware molecular pI/charge/MW values are claimed, and these metrics remain excluded from primary cyclic comparisons.</p>"

html += "<h2>Statistical Uncertainty and Robustness Audit</h2>"
html += "<p class='note'>Bootstrap 95% percentile intervals use peptide-level resampling of published candidates and independent resampling of selected reference sequences (2,000 iterations, seed 42). These intervals and exploratory ordering frequencies are conditional on available records; they cannot correct publication/selection bias. Rank frequencies do not quantify overall model performance. n>=3 is a computational threshold, not a power guarantee. Spearman correlations and scaling/weight scenarios are exploratory. Models with fewer than 3 candidates remain descriptive only.</p>"
html += "<h3>Bootstrap Distance Intervals and Observed Candidate Counts</h3>" + bootstrap_df.round(4).to_html(index=False)
html += "<h3>Exploratory Bootstrap Ordering Frequencies (supplementary diagnostic only; not performance ranks)</h3>" + bootstrap_rank_df.round(4).to_html(index=False)
html += "<h3>Per-Descriptor Effect Sizes and Bootstrap 95% Intervals</h3>" + feature_effect_ci_df.round(4).to_html(index=False)
html += "<h3>All Model Sample Sizes, Including Excluded Small Groups</h3>" + model_sample_audit_df.to_html(index=False)
html += "<h3>Feature Redundancy (Spearman)</h3>" + correlation_df.round(4).to_html(index=False)
html += "<h3>Normalization / Weighting / MW Exclusion Sensitivity</h3>" + robustness_df.round(4).to_html(index=False)
html += "<h2>FASTA Input Integrity, Exact Overlap and Within-Model Similarity Audit</h2>"
html += "<p class='note'>All FASTA inputs were checked for normalized exact-sequence duplicates (including pre-filter input counts), exact overlaps between distinct source/model groups, and within-model high similarity using normalized Levenshtein edit similarity (threshold 0.80). No observations were removed by this diagnostic audit. Exact uniqueness is not evidence of peptide independence; common generative ancestry and selection processes may remain.</p>"
html += "<h3>Original Input Counts and Validation Exclusions</h3>" + input_integrity_df.to_html(index=False)
html += "<h3>Duplicate and Within-Model Similarity Summary</h3>" + duplicate_summary_df.to_html(index=False)
html += "<h3>Exact Duplicates Within Groups</h3>" + within_duplicate_df.to_html(index=False)
html += "<h3>Exact Overlaps Between Models / Generated and References / Reference Sets</h3>" + exact_overlap_df.to_html(index=False)
html += "<h3>Within-Model High Sequence Similarity (>=0.80 normalized edit similarity)</h3>" + near_similarity_df.to_html(index=False)
html += "<h3>Within-Model Similarity Pairs (>=0.70; exploratory)</h3>" + near_similarity_070_df.to_html(index=False)
html += "<h3>Exploratory One-Per-Cluster Sensitivity: 0.80 and 0.70 Thresholds</h3>" + cluster_sensitivity_counts_df.to_html(index=False)
html += "<h3>Descriptive Composite Distance Before and After Cluster Reduction</h3>" + composite_cluster_sensitivity_df.round(4).to_html(index=False)
html += "<h3>Descriptor Distance and BH q-Value Sensitivity</h3>" + cluster_sensitivity_comparisons_df.round(4).to_html(index=False)
html += "<p class='note'>Thresholds 0.80 and 0.70 are exploratory edit-similarity heuristics, not validated biological or statistical dependence thresholds. One representative per within-model connected component was retained for sensitivity calculations only; the original dataset and primary analyses were unchanged. Only comparisons retaining at least 3 sequences were retested. Shared generation runs, ancestral templates, and study-level selection can induce dependence even among dissimilar sequences. Therefore p-values and peptide-bootstrap intervals remain exploratory/conditional.</p>"

html += "<h2>Model-Level Analysis</h2>"
html += "<p class='note'>KDE and ECDF panels labelled EXPLORATORY (Cyclic MW, pI, NetCharge) are sequence-only diagnostics and are excluded from significance testing and descriptive composite-distance comparisons.</p>"
html += "<h3>Plot 1 - Distribution Overlay (KDE)</h3>" + img_dist
html += "<h3>Plot 2 - Empirical CDF (ECDF)</h3>" + img_ecdf
html += "<h3>Plot 3 - Mann-Whitney U: BH-FDR-adjusted q-values (not an equivalence test)</h3>" + img_sig
html += "<h3>Plot 4 - Descriptive Reference-Percentile Coverage (alphabetical) (selected published candidates)</h3>" + img_rank
html += "<h3>Full Descriptive Model Table (alphabetical within category; no competitive ranks)</h3>"
html += rl_rank_df[["Category", "Model", "N_sequences", "Low_N_Flag",
                     "Properties_Tested", "Composite_Compliance_%", "Composite_Distance",
                     "Distance_CI_Lower_95", "Distance_CI_Upper_95"]].to_html(index=False)
html += "<h3>Full Statistical Comparison Table (Model vs Selected database reference, per Property)</h3>" + comparison_df.to_html(index=False)

html += "<h2>Generation-Strategy-Level Analysis (RL vs Pseudo-RL vs Hybrid and Advanced)</h2>"
html += "<h3>Plot 5 - Descriptive Strategy Coverage (alphabetical) (study-selection confounded)</h3>" + img_strategy
html += "<h3>Descriptive Strategy Table (not comparable performance ranking)</h3>" + rank_df_strategy.to_html(index=False)
html += "<h3>Full Statistical Comparison Table (Strategy vs Selected database reference, per Property)</h3>" + comparison_df_strategy.to_html(index=False)


# Sequence novelty supplement is independent of physicochemical benchmarking.
assert "reference_sequence_similarity_summary_df" in globals(), "Run new Cell 4C before generating the report."
html += "<h2>Generated-vs-Selected-Reference Sequence Similarity: Local Alignment</h2>"
html += ("<p class='note'>Exploratory Smith-Waterman local alignment with BLOSUM62 (gap open -10, "
         "extension -0.5), minimum 70% query coverage and at least five aligned amino-acid pairs. "
         "Identity is the matching fraction of aligned residue pairs; not BLAST or CD-HIT. "
         "The comparisons use only selected category-specific reference subsets, not full databases "
         "or model training sets. A high local identity is not a validated measure of peptide novelty. "
         "Unqualified alignments are counted separately and never treated as novel. "
         "Only head-to-tail CYC_BUILDER candidate rotations are considered; no ring/topology "
         "equivalence is assumed for disulfide-closed HIGHPLAY. Chemical modifications are unavailable. "
         "Reference leave-one-out and composition-shuffled sequences provide exploratory baseline comparisons.</p>")
html += "<h3>Generated sequences: by-model local alignment summary</h3>" + reference_sequence_similarity_summary_df.round(4).to_html(index=False)
html += "<h3>Composition-shuffled generated sequences: by-model control</h3>" + reference_similarity_shuffled_summary_df.round(4).to_html(index=False)
html += "<h3>Selected-reference leave-one-out: category control</h3>" + reference_similarity_loo_summary_df.round(4).to_html(index=False)
html += "<h3>Per-generated-sequence nearest local reference alignment</h3>" + reference_sequence_similarity_df.round(4).to_html(index=False)

html += f"<h2>All Peptides ({len(df)} sequences)</h2>"
html += df.to_html(index=False)

html += """
<h2>Method Notes</h2>
<ul>
  <li><b>Literature-defined ranges:</b> selected task-specific reported thresholds only; no universal pI compliance range is imposed.</li>
  <li><b>Cyclic chemistry:</b> CycPeptMPDB FASTA represents mapped amino acid identity only, not connectivity, stereochemistry, termini, or modifications. Thus primary cyclic statistical tests and descriptive composite-distance comparisons are based on Length, GRAVY and AliphaticIndex only. Sequence-derived MW, pI and NetCharge remain exploratory and are not interpreted as actual molecular values. Head-to-tail closure removes free terminal ionization; disulfide closure does not itself remove termini, and actual terminal caps and linked residues matter. No topology-specific values were fabricated.</li>
  <li><b>Multiple comparisons:</b> BH-FDR applied separately for Mann-Whitney and KS within each peptide category, across eligible model-by-descriptor comparisons; separate test families are used for strategy-level analyses. Non-significance does NOT prove similarity or equivalence.</li>
  <li><b>Ranking and uncertainty:</b> Smaller observed composite distance denotes only greater descriptive similarity of these selected sequence sets to the specified reference on included descriptors; overlapping bootstrap intervals and descriptor-sensitive differences do not establish statistically or biologically superior generative models.</li>
  <li><b>Environment:</b> Biopython 1.86 is required and checked at run time. Reference FASTA files are externally constructed inputs; integrity checks do not independently reproduce database acquisition, filtering, or sampling. Please retain the source files, metadata and versioned construction scripts separately.</li>
  <li><b>Selection caveat:</b> Dataset includes available, sequence-identifiable, literature-reported candidates, not standardized random samples from full generator outputs. Published candidate selection and validation vary across studies.</li>
  <li><b>Redundancy:</b> Length/MW may correlate, so six marginal descriptor distances do not constitute six independent lines of evidence. Normalization by a small reference SD can magnify an attribute.</li>
  <li><b>Independent runs:</b> The dataset contains reported candidate sequences rather than reproducible standardized outputs from the underlying model weights, training configurations, and common sampling protocols. No independent model-generation runs were performed; the supplied code only recomputes analysis on available candidate sets.</li>
  <li><b>Scaling applicability:</b> For Cyclic, the primary descriptors are Length, GRAVY and AliphaticIndex, so MW-exclusion and Length/MW-downweighting scenarios do not apply; reference-IQR scaling is evaluated instead.</li>
  <li><b>Per-property effect sizes:</b> Normalized Wasserstein distance and bootstrap 95% percentile intervals are reported per eligible descriptor; estimates are withheld for n&lt;5 and remain exploratory for low N.</li>
  <li><b>Small samples:</b> Rug marks replace smooth KDE for model groups with fewer than 10 observations; ECDF conveys observed discrete sample sizes. n>=3 does not assure sufficient power.</li>
  <li><b>Mann-Whitney U test:</b> non-parametric test for whether values are drawn from the same distribution
  as the selected category-specific database reference subset.</li>
  <li><b>Kolmogorov-Smirnov test:</b> tests for differences in overall distribution shape/location.</li>
  <li><b>Wasserstein distance (normalized by selected reference-set SD):</b> continuous measure of distance; 0 = identical.</li>
  <li><b>Compliance %:</b> percentage of values falling inside the 5th-95th percentile range of the selected category-specific reference peptides.</li>
  <li><b>Model vs. Strategy:</b> Category is biological; Strategy is a generation-method dimension, tracked
  independently since one strategy can produce peptides across multiple categories.</li>
</ul>
</body></html>
"""

output = "peptide_benchmarking_report.html"
with open(output, "w", encoding="utf-8") as f:
    f.write(html)

files.download(output)
print(f"\nReport saved: {output}")
print("\n--- Descriptive Model Sample Sizes ---")
print(model_sample_audit_df.to_string(index=False))
print("\n--- Descriptive Strategy Comparison ---")
print(rank_df_strategy[["Category","Strategy","N_sequences","Composite_Compliance_%"]].to_string(index=False))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Report saved: peptide_benchmarking_report.html

--- Descriptive Model Sample Sizes ---
Category       Model  N_sequences                              Inference_Status  Low_N_Flag
     AMP      AMP-RL            6          Exploratory tests; power not assured LOW N (<10)
     AMP   AMPAINTER           30          Exploratory tests; power not assured            
     AMP    DIFF-AMP            6          Exploratory tests; power not assured LOW N (<10)
     AMP       HMAMP           10          Exploratory tests; power not assured            
     AMP        MOQA            4          Exploratory tests; power not assured LOW N (<10)
     AMP      PEPGAN            6          Exploratory tests; power not assured LOW N (<10)
     AMP      RR-ADS           15          Exploratory tests; power not assured            
  Cyclic CYC_BUILDER            9          Exploratory tests; power not assured LOW N (<10)
  Cyclic    HIGHPLAY           30          Exploratory tests; power not assured     